---
execute:
  enabled: true
---

[![](imagens/colab-badge.png){width="16%"}](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/apendice_f/apendice_f_aluno.ipynb)
[![](imagens/github-badge.png){width="19%"}](https://github.com/fzampirolli/pdi-vc)

# Análise de Percepção e Avaliação Didática: PDI-VC 2026.2 {#apendice-pdi-vc-2026-2}

Este apêndice apresenta uma análise **quantitativa e qualitativa, agregada e não identificável**, da percepção dos estudantes sobre a experiência pedagógica na disciplina de **Processamento Digital de Imagens e Visão Computacional (PDI-VC)**, oferecida no segundo quadrimestre de 2026. A análise contempla avaliações estruturadas, comparações entre os questionários **PRÉ e PÓS, de participação voluntária**, e manifestações espontâneas registradas nas questões abertas.

Os resultados foram organizados de modo a preservar a **confidencialidade dos participantes**. Não são divulgados nomes, endereços eletrônicos, respostas individualizadas ou quaisquer outros elementos que permitam a identificação direta ou indireta dos estudantes.

::: {.callout-note}

## Proteção da confidencialidade, tratamento dos dados e reprodutibilidade {.unnumbered}

As análises foram realizadas exclusivamente sobre **dados agregados ou anonimizados**, utilizados para a avaliação e análise da experiência pedagógica. Os resultados são apresentados coletivamente, sem associação entre respostas e identidades individuais.

As tabelas, estatísticas e gráficos foram gerados de forma **automática e reprodutível** a partir dos dados utilizados na análise. O *notebook* Colab, disponibilizado pelo ícone no canto superior esquerdo, documenta os procedimentos computacionais empregados, incluindo o processamento dos dados, as transformações realizadas e a geração dos resultados apresentados neste apêndice.

Os **dados brutos contendo informações potencialmente identificáveis não são publicados**, incorporados ao material de divulgação ou disponibilizados por meio do *notebook*. Da mesma forma, respostas individuais não são apresentadas de modo que possam ser associadas a participantes específicos.

Eventuais materiais necessários à **auditoria metodológica** permanecem sob guarda do responsável pela análise e poderão ser disponibilizados, quando cabível, em condições compatíveis com as exigências éticas aplicáveis e sem exposição de dados pessoais ou identificáveis.

Para a análise longitudinal, quando necessária, foi utilizado um identificador exclusivamente para o **pareamento entre as respostas PRÉ e PÓS**. Esse identificador não é divulgado. Após a conclusão dos procedimentos que dependem do pareamento, os resultados são apresentados apenas de forma agregada, preservando a confidencialidade dos participantes.

:::

## Caracterização da Amostra e Metodologia de Coleta

A coleta de dados ocorreu de forma **voluntária** em dois momentos do quadrimestre. O **questionário PRÉ-curso**, aplicado na primeira semana de aula, caracterizou o perfil dos estudantes e suas percepções iniciais. O **questionário PÓS-curso**, aplicado ao final do quadrimestre, juntamente com a prova final, avaliou a experiência na disciplina após a utilização dos recursos didáticos e das estratégias analisadas neste apêndice. A distribuição dos respondentes entre as três turmas é apresentada na @tbl-contagem-turmas.

::: {.callout-note}

### Nota metodológica {.unnumbered}

Os testes estatísticos identificam **diferenças e associações**, mas não estabelecem relações causais. Portanto, diferenças entre recursos que utilizaram ou não IA não devem ser atribuídas isoladamente à tecnologia, pois esses recursos também diferem quanto à finalidade pedagógica, ao formato, ao nível de interação e ao momento de utilização.

:::


In [1]:
#| include: false  # esconde código E output, mas ainda executa (útil pra imports/setup)
#| echo: false
#| output: true

import os
import re
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
import seaborn as sns

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# --------------------------------------------------------------------------
# CONFIGURAÇÃO: caminhos dos arquivos CSV locais na pasta dataset/
# --------------------------------------------------------------------------
PATH_POS = os.path.join("dataset", "POS_anonimizado.csv")
PATH_PRE = os.path.join("dataset", "PRE_anonimizado.csv")

# ==============================================================================
# PADRÃO VISUAL DO NOTEBOOK — usado em TODOS os gráficos a partir daqui.
# Qualquer ajuste de cor/estilo deve ser feito aqui, uma única vez.
# ==============================================================================
CORES = {
    "pre":            "#91bfdb",   # respostas do momento PRÉ
    "pos":            "#a6d96a",   # respostas do momento PÓS 
    "aprovado":       "#2b5c8f",   # item significativo / aprovado (azul escuro) 
    "neutro":         "#8c96c6",   # item não significativo (roxo acinzentado) 
    "linha_neutra":   "#d73027",   # linha de referência do ponto neutro (3,0) 
    "media_marker":   "#d73027",   # marcador de média em boxplots 
    "outlier":        "#e67e22",   # destaque de casos excluídos/atípicos 
    "likert_5cores":  ["#d7191c", "#fdae61", "#ffffbf", "#a6d96a", "#1a9641"],  # notas 1→5 
    "efeito_pequeno": "#3498db", 
    "efeito_medio":   "#e67e22", 
    "efeito_grande":  "#27ae60", 
    "efeito_nulo":    "#7f8c8d", 
    "heatmap_cmap":   "YlGnBu_r", 
    "texto_escuro":   "#111111", 
    "texto_medio":    "#333333", 
}
FONTE = {"titulo": 13, "subtitulo": 12, "eixo": 11, "texto": 9.5, "legenda": 9} 

# Ponto neutro da escala Likert de 5 pontos — referência única para todos
# os testes de uma amostra (Wilcoxon vs. neutro) usados ao longo do apêndice.
VALOR_NEUTRO = 3.0

sns.set_theme(style="whitegrid", context="notebook", palette="muted") 
plt.rcParams["figure.figsize"] = (10, 6) 
plt.rcParams["font.size"] = 10 
plt.rcParams["axes.titleweight"] = "bold" 

def extrair_descricao(col, max_len=90):
    """Extrai prioritariamente o texto contido dentro de colchetes [...] do cabeçalho do Forms.
    Se não houver colchetes com texto descritivo, limpa o texto do enunciado."""
    # Procura por conteúdos dentro de colchetes [...]
    m = re.findall(r'\[(.*?)\]', str(col))
    if m:
        # Filtra códigos simples como [Q08_02] para pegar o texto/rótulo completo do colchete
        # Exemplo: de "[Q09_02 - Simuladores]" pega "Q09_02 - Simuladores"
        textos = [t.strip() for t in m if len(t.strip()) > 0]
        if textos:
            # Seleciona o trecho entre colchetes mais informativo (o maior ou mais descritivo)
            desc = sorted(textos, key=len, reverse=True)[0]
            if len(desc) > max_len:
                desc = desc[:max_len - 1].rstrip() + "…"
            return desc

    # Fallback (caso não haja colchetes na coluna)
    desc = re.sub(r"\[+(PRE_Q\d+|Q\d+(?:_\d+)?)\]*", "", str(col)).strip(" -–:\u200b\"")
    desc = re.sub(r"^\d{1,2}[\.\)]\s*", "", desc)
    desc = re.sub(r"\s+", " ", desc).strip()
    if len(desc) > max_len:
        desc = desc[:max_len - 1].rstrip() + "…"
    return desc

def extrair_codigo(col):
    """Extrai um código curto tipo Q01, Q08_05, PRE_Q06 do cabeçalho longo do Forms."""
    m = re.search(r"\[+(PRE_Q\d+|Q\d+(?:_\d+)?)", col)
    if m:
        return m.group(1)
    if col.strip() == "" or col.startswith("[Q09"):
        return None  # coluna de checkbox sem rótulo (bug identificado) -> tratada à parte
    return col  # mantém original (ex.: Carimbo, Turma, e-mail)

def estilo_eixos(ax):
    """Acabamento padrão de todos os gráficos: remove as bordas superior e direita.""" 
    ax.spines["top"].set_visible(False) 
    ax.spines["right"].set_visible(False) 
    return ax 


def linha_neutra(ax, valor=3.0, orientacao="v", label="Ponto Neutro (3,0)"):
    """Desenha a linha de referência do ponto neutro da escala, com o estilo padrão.""" 
    if orientacao == "v": 
        ax.axvline(valor, color=CORES["linha_neutra"], linestyle="--", linewidth=1.5, alpha=0.85, label=label) 
    else:
        ax.axhline(valor, color=CORES["linha_neutra"], linestyle="--", linewidth=1.5, alpha=0.85, label=label) 

print("Configuração inicial do notebook concluída. Imports, funções utilitárias e estilo visual carregados.")

Configuração inicial do notebook concluída. Imports, funções utilitárias e estilo visual carregados.


In [2]:
#| include: false  # esconde código E output, mas ainda executa (útil pra imports/setup)
#| echo: false
#| output: true

# --------------------------------------------------------------------------
# Opções de chunk usadas aqui:
#   echo: false     -> esconde o código, mantém o output (tabela/gráfico)
#   include: false  -> esconde código E output, mas ainda executa (setup/imports)
#   output: false   -> mostra o código, esconde só o output
# --------------------------------------------------------------------------

import json
import os

# ==============================================================================
# 1. CARREGAMENTO DOS DADOS
# ==============================================================================
def carregar_pipeline():
    """Carrega PRÉ e PÓS anonimizados do zero e normaliza a coluna Turma
    (funde grafias divergentes como 'CCM'/'MCC')."""

    def _carregar(caminho_csv):
        df = pd.read_csv(caminho_csv)
        df.columns = [str(c).strip() for c in df.columns]
        return df

    def _normalizar_turmas(df, coluna="Turma", mapeamento=None):
        """Normaliza grafias divergentes da coluna Turma (ex.: 'CCM' e 'MCC' são a
        mesma turma, só com a sigla do curso invertida). Aplica strip + upper antes
        de mapear, para pegar também variações de espaço/caixa (' mcc', 'Mcc' etc.)."""
        if mapeamento is None:
            mapeamento = {"CCM": "MCC"}  # ajuste aqui se o canônico deveria ser CCM
        df = df.copy()
        turma_normalizada = df[coluna].astype(str).str.strip().str.upper()
        df[coluna] = turma_normalizada.replace(mapeamento)
        return df

    pos = _carregar(PATH_POS)
    pre = _carregar(PATH_PRE)
    pos = _normalizar_turmas(pos)
    pre = _normalizar_turmas(pre)

    print("PÓS -> colunas:", pos.shape[1], "| respondentes:", pos.shape[0])
    print("PRÉ -> colunas:", pre.shape[1], "| respondentes:", pre.shape[0])

    return pos, pre


pos, pre = carregar_pipeline()

# ==============================================================================
# 1b. AMOSTRA PAREADA PRÉ x PÓS
# ------------------------------------------------------------------------
# Calculada uma única vez aqui e reaproveitada em todo o apêndice (Wilcoxon
# pareado, textos inline etc.), em vez de recalculada célula a célula.
# ==============================================================================
pareado = pre.merge(pos, on="ID_Anonimo", suffixes=("_pre", "_pos"))
n_pareados = len(pareado)
assert n_pareados == 15, f"Esperava 15 estudantes pareados, encontrei {n_pareados}."

# Dimensões comparadas entre PRÉ e PÓS: rótulo, coluna no PRÉ, coluna no PÓS
# e slug curto (usado para nomear as variáveis escalares exportadas ao texto,
# ex.: media_pre_prog, mw_d_ia). Definido uma única vez e reaproveitado nas
# células de Wilcoxon pareado (@tbl-pre-pos-wilcoxon) e Mann-Whitney (@tbl-pre-pos-mw).
MAPEAMENTO_PRE_POS = [
    ("Capacidade de programação", "PRE_Q01", "Q01", "prog"),
    ("Feedback por IA",           "PRE_Q02", "Q06", "ia"),
    ("Atuação acadêmica",         "PRE_Q03", "Q02", "acad"),
    ("Atuação profissional",      "PRE_Q04", "Q03", "prof"),
    ("Fundamentos matemáticos",   "PRE_Q05", "Q04", "mat"),
    ("Autonomia com bibliotecas", "PRE_Q06", "Q05", "auto"),
]

# ==============================================================================
# 2. METADADOS DAS PERGUNTAS (rótulos + enunciados completos)
# ==============================================================================
# Gerado pelo sigilo.py a partir dos headers originais do Forms (antes da
# anonimização). É a única fonte de verdade; não há mais extração manual nem
# tentativa de extrair dos CSVs anonimizados, já que dataset/*_anonimizado.csv
# só tem colunas curtas (Q01, Q02...), sem texto de header sobrando.
with open(os.path.join("dataset", "metadados_perguntas.json"), encoding="utf-8") as f:
    METADADOS_PERGUNTAS = json.load(f)

# Overrides manuais — só para itens que o Forms não etiqueta com rótulo entre
# colchetes (perguntas abertas de texto livre). Deixe vazio se não houver
# nenhuma exceção pendente.
DESCRICOES_OVERRIDE = {
    "Q14": "Maior contribuição da IA",
    "Q15": "Dificuldades e ajustes metodológicos",
}

DESCRICOES = {
    cod: DESCRICOES_OVERRIDE.get(cod) or info.get("rotulo") or cod
    for cod, info in METADADOS_PERGUNTAS.items()
}
ENUNCIADOS_COMPLETOS = {
    cod: info.get("enunciado")
    for cod, info in METADADOS_PERGUNTAS.items()
    if info.get("enunciado")
}

print(f"{len(DESCRICOES)} rótulos e {len(ENUNCIADOS_COMPLETOS)} enunciados carregados de metadados_perguntas.json.")
assert len(DESCRICOES) >= 25, f"Só {len(DESCRICOES)} rótulos disponíveis — confira dataset/metadados_perguntas.json."

# ==============================================================================
# 3. FUNÇÕES AUXILIARES DE TABELA
# ==============================================================================
def tabela_itens_md(codigos, coluna_texto="Enunciado aplicado"):
    """Monta um DataFrame Item/Texto para uma lista de códigos, usando os
    rótulos e enunciados já carregados de metadados_perguntas.json."""
    linhas = [
        {"Item": f"`{cod}`", coluna_texto: f"**{DESCRICOES.get(cod, cod)}:** {ENUNCIADOS_COMPLETOS.get(cod, '—')}"}
        for cod in codigos
    ]
    return pd.DataFrame(linhas)


def extrair_opcoes_q09(serie_q09):
    """Varre as respostas de Q09 (texto livre com múltiplas seleções, no formato
    '[Q09_XX - Rótulo] Descrição, [Q09_YY - Rótulo] Descrição, ...') e monta
    automaticamente código -> (rótulo curto, descrição completa), agregando o
    conjunto de opções que aparece em qualquer linha de resposta."""
    opcoes = {}
    for valor in serie_q09.dropna():
        partes = re.split(r",\s*(?=\[Q09_)", str(valor))
        for parte in partes:
            m = re.match(r"\[(Q09_\d{2})\s*-\s*([^\]]+)\]\s*(.*)", parte.strip())
            if m:
                cod, rotulo, desc = m.groups()
                if cod not in opcoes:
                    opcoes[cod] = (rotulo.strip(), desc.strip().rstrip(","))
    return opcoes

def fmt_num_br(x, casas=3):
    if pd.isna(x):
        return "—"
    if casas == 0 or float(x).is_integer():
        res = str(int(round(x)))
    else:
        res = f"{x:.{casas}f}".replace(".", ",")
    return res.replace("-", "−")


def fmt_p_br(p):
    if pd.isna(p):
        return "—"
    return "<0,001" if p < 0.001 else f"{p:.3f}".replace(".", ",")

OPCOES_Q09 = extrair_opcoes_q09(pos["Q09"])
print(f"{len(OPCOES_Q09)} opções de Q09 identificadas automaticamente (esperado: 10).")

display(pos.head())

PÓS -> colunas: 28 | respondentes: 80
PRÉ -> colunas: 20 | respondentes: 104
31 rótulos e 31 enunciados carregados de metadados_perguntas.json.
10 opções de Q09 identificadas automaticamente (esperado: 10).


,ID_Anonimo,Carimbo de data/hora,Turma,Q01,Q02,Q03,Q04,Q05,Q06,Q07,Q08_01,Q08_02,Q08_03,Q08_04,Q08_05,Q08_06,Q08_07,Q08_08,Q08_09,Q08_10,Q08_11,Q09,Q10,Q11,Q12,Q13,Q14,Q15
0,Aluno_NA_POS_01,09/08/2026 20:00:04,DA1,3,4,4,3,3,5,4,4,3,4,5,5,3,3,3,1,4,5,[Q09_03 - EPs em Aula] Resolução dos Exercício...,4,3,3,3,NaN,NaN
1,Aluno_POS_01,09/08/2026 20:06:08,DA2,4,3,4,4,5,5,4,5,5,5,4,5,5,5,3,5,5,5,[Q09_02 - Simuladores] Simuladores interativos...,4,4,4,4,NaN,NaN
2,Aluno_POS_02,09/08/2026 20:11:04,DA2,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,[Q09_02 - Simuladores] Simuladores interativos...,5,4,4,4,NaN,NaN
3,Aluno_NA_POS_02,09/08/2026 20:11:44,DA2,5,1,1,2,3,4,3,2,1,2,4,4,3,4,3,4,4,5,[Q09_03 - EPs em Aula] Resolução dos Exercício...,3,3,3,3,NaN,NaN
4,Aluno_POS_03,09/08/2026 20:13:25,DA1,5,5,5,3,4,3,5,4,2,5,5,4,4,4,3,5,5,4,[Q09_02 - Simuladores] Simuladores interativos...,3,5,2,1,NaN,NaN


In [3]:
#| quarto-raw: false
#| label: tbl-contagem-turmas
#| tbl-cap: "Quantidade de respondentes por turma nos questionários PRÉ e PÓS."
#| echo: false
#| output: true

import pandas as pd
from IPython.display import Markdown

# 1. Contagem de respondentes por turma
contagem_pre = pre["Turma"].value_counts()
contagem_pos = pos["Turma"].value_counts()

turmas = sorted(set(contagem_pre.index) | set(contagem_pos.index))

# 2. Monta as linhas da tabela
resultados_turmas = [
    {
        "Turma": turma,
        "PRÉ (n)": int(contagem_pre.get(turma, 0)),
        "PÓS (n)": int(contagem_pos.get(turma, 0)),
    }
    for turma in turmas
]

# 3. Totais e variáveis escalares para uso inline no texto
total_pre = sum(r["PRÉ (n)"] for r in resultados_turmas)
total_pos = sum(r["PÓS (n)"] for r in resultados_turmas)

# Dicionários auxiliares
n_pre = {r["Turma"]: r["PRÉ (n)"] for r in resultados_turmas}
n_pos = {r["Turma"]: r["PÓS (n)"] for r in resultados_turmas}

# Variáveis diretas por turma (evita erro de sintaxe no inline)
pre_da1 = n_pre.get("DA1", 0)
pre_da2 = n_pre.get("DA2", 0)
pos_da2 = n_pos.get("DA2", 0)
pre_mcc = n_pre.get("MCC", 0)
pos_mcc = n_pos.get("MCC", 0)

# 4. Linha de Total para a tabela
resultados_turmas.append(
    {
        "Turma": "**Total**",
        "PRÉ (n)": f"**{total_pre}**",
        "PÓS (n)": f"**{total_pos}**",
    }
)

df_turmas = pd.DataFrame(resultados_turmas)
Markdown(
    df_turmas.to_markdown(index=False, colalign=("left", "center", "center"))
)

| Turma     |  PRÉ (n)  |  PÓS (n)  |
|:----------|:---------:|:---------:|
| DA1       |    39     |    39     |
| DA2       |    46     |    30     |
| MCC       |    19     |    11     |
| **Total** |  **104**  |  **80**   |

Foram obtidas **`{python} total_pre` respostas no PRÉ** e **`{python} total_pos` no PÓS**. A partir das informações fornecidas voluntariamente pelos estudantes para possibilitar a identificação entre os dois momentos, foram identificados **`{python} n_pareados` estudantes que responderam a ambos os questionários**, constituindo a **amostra pareada** utilizada na análise da evolução individual.

As turmas **DA1** e **DA2** correspondem à graduação, oferecida no período **matutino**, enquanto a **MCC** corresponde à pós-graduação, oferecida no período **vespertino**. Entre o PRÉ e o PÓS, o número de respostas **manteve-se em `{python} pre_da1` na DA1**, enquanto diminuiu de **`{python} pre_da2` para `{python} pos_da2` na DA2** e de **`{python} pre_mcc` para `{python} pos_mcc` na MCC**. Assim, a redução da participação concentrou-se nas turmas **DA2 e MCC**, enquanto a **DA1 permaneceu estável**.

Considerando a natureza ordinal das escalas *Likert* de cinco pontos, foram empregados **testes estatísticos não paramétricos**. O teste de *Mann--Whitney U* [@mann1947] foi utilizado para comparações entre amostras independentes; o teste de *Kruskal--Wallis* [@kruskal1952], para comparações entre mais de dois grupos independentes; e o teste de Friedman [@friedman1937], para comparar diferentes recursos avaliados pelos mesmos estudantes. Na comparação pareada entre PRÉ e PÓS, empregou-se o teste de *Wilcoxon* [@wilcoxon1945].

Como complemento aos testes de significância, foram considerados **tamanhos de efeito**, correlações de *Spearman*, consistência interna pelo $\alpha$ de *Cronbach* e **preferências declaradas**, permitindo caracterizar os resultados sob diferentes perspectivas analíticas.


## Itens do questionário PÓS

O questionário PÓS foi estruturado em cinco blocos. O **Bloco 1** reúne a identificação opcional do estudante e a seleção de uma das três turmas. O **Bloco 2**, apresentado na @tbl-pos-bloco2, aborda a **autopercepção de aprendizado e desempenho**. O **Bloco 3**, apresentado na @tbl-pos-bloco3 (`Q08`) e na @tbl-pos-q09 (`Q09`), avalia os **recursos didáticos**. O **Bloco 4**, apresentado na @tbl-pos-bloco4, investiga a percepção sobre a **integração da Inteligência Artificial (IA)** ao processo de ensino e aprendizagem. Por fim, o **Bloco 5**, apresentado na @tbl-pos-bloco5, reúne duas questões abertas.

Os itens avaliativos dos Blocos 2, 3 e 4 utilizam uma **escala *Likert* de cinco pontos**: 1 = *Discordo totalmente*; 2 = *Discordo*; 3 = *Indiferente*; 4 = *Concordo*; e 5 = *Concordo totalmente*. A questão `Q09`, pertencente ao Bloco 3, adota formato distinto, solicitando a **seleção de exatamente quatro recursos** considerados mais eficazes para a aprendizagem pelo estudante.


### Itens do Bloco 2 — Autopercepção de Aprendizado e Desempenho

A @tbl-pos-bloco2 apresenta os sete itens do Bloco 2, que avaliam a **autopercepção dos estudantes sobre o aprendizado e o desempenho** ao final da disciplina. Os seis primeiros itens abrangem capacidade de programação, aplicação dos conhecimentos em outras disciplinas e na atuação profissional, fundamentos matemáticos, autonomia no uso de bibliotecas e percepção sobre o *feedback* por IA. Esses itens possuem correspondência com questões do questionário **PRÉ** e serão analisados de forma pareada na seção seguinte, permitindo avaliar a evolução das percepções individuais. O item `Q07` apresenta uma avaliação global da evolução dos conhecimentos teóricos e práticos em PDI-VC ao longo da disciplina.

In [4]:
#| label: tbl-pos-bloco2
#| tbl-cap: "Itens do Bloco 2 — Autopercepção de Aprendizado e Desempenho."
#| tbl-colwidths: "[8,80]"
#| echo: false
#| output: true

codigos_bloco2 = ["Q01", "Q02", "Q03", "Q04", "Q05", "Q06", "Q07"]
df_bloco2 = tabela_itens_md(codigos_bloco2)
Markdown(df_bloco2.to_markdown(index=False, colalign=("left", "left")))

| Item   | Enunciado aplicado                                                                                                                                                                                      |
|:-------|:--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `Q01`  | **PÓS - Capacidade de Programação:** Atualmente, consigo resolver problemas computacionais básicos usando programação.                                                                                  |
| `Q02`  | **PÓS - Atuação Acadêmica:** Os conhecimentos desenvolvidos nesta disciplina têm me ajudado na minha atuação acadêmica em outras disciplinas.                                                           |
| `Q03`  | **PÓS - Atuação Profissional:** Os conhecimentos desenvolvidos nesta disciplina serão úteis para a minha atuação profissional                                                                           |
| `Q04`  | **PÓS - Fundamentos Matemáticos:** Atualmente, considero que minha base em Álgebra Linear e Cálculo (conceitos usados em PDI) é muito boa.                                                              |
| `Q05`  | **PÓS - Autonomia com Bibliotecas:** Atualmente, sinto que tenho boa autonomia para utilizar bibliotecas de processamento de dados/imagens (ex: OpenCV, NumPy, Matplotlib, morph.py).                   |
| `Q06`  | **PÓS - Feedback por IA:** Atualmente, considero útil o feedback automático de código gerado por IA.                                                                                                    |
| `Q07`  | **GLOBAL_BLOCO2 - Avaliação Geral do Aprendizado:** De forma geral, considero que o meu nível de conhecimento teórico e prático na área de PDI-VC evoluiu significativamente ao longo desta disciplina. |

### Itens do Bloco 3 — Recursos Didáticos

O **Bloco 3** avalia os recursos didáticos utilizados na disciplina. É composto por dez itens de avaliação (`Q08_01`--`Q08_10`), apresentados na @tbl-pos-bloco3; uma questão sobre o acesso ao VPL (`Q08_11`); uma questão de **preferência pelos recursos** (`Q09`); e uma **avaliação global** (`Q10`).


In [5]:
#| label: tbl-pos-bloco3
#| tbl-cap: "Itens do Bloco 3 — Recursos Didáticos."
#| tbl-colwidths: "[8,80]"
#| echo: false     # esconde o código, mas mantém o output (tabela/gráfico)

codigos_bloco3 = [f"Q08_{i:02d}" for i in range(1, 12)] + ["Q09", "Q10"]
df_bloco3 = tabela_itens_md(codigos_bloco3)
Markdown(df_bloco3.to_markdown(index=False, colalign=("left", "left")))

| Item     | Enunciado aplicado                                                                                                                                                                                                                                                                       |
|:---------|:-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `Q08_01` | **Livro Interativo:** O uso do Livro Interativo (HTML/IPYNB/PDF) com simuladores e teoria integrada contribuiu significativamente para o meu aprendizado.                                                                                                                                |
| `Q08_02` | **Material Didático por IA:** Os vídeos curtos (~7 min) e os slides (~12 slides) produzidos via Gemini Notebook ajudaram a introduzir e sintetizar o conteúdo antes do início da aula prática no Colab                                                                                   |
| `Q08_03` | **Simuladores Interativos:** A manipulação visual de parâmetros em tempo real nos simuladores interativos (no HTML/Colab) facilitou a compreensão dos algoritmos.                                                                                                                        |
| `Q08_04` | **Resolução de EPs em Aula:** A resolução acompanhada dos Exercícios de Programação (EPs) durante o horário de aula foi fundamental para o meu desempenho prático.                                                                                                                       |
| `Q08_05` | **Biblioteca morph.py:** O uso da biblioteca didática morph.py (com implementações didáticas como mm.dil0 e otimizadas) tornou o código dos algoritmos mais transparente e compreensível.                                                                                                |
| `Q08_06` | **Provas e Simulados Paramétricos:** As avaliações parametrizadas pelo MCTest e aplicadas via SEB (com questões individuais) foram eficazes para avaliar meu aprendizado de forma justa.                                                                                                 |
| `Q08_07` | **Feedback Socrático no VPL:** O feedback qualitativo/socrático gerado por IA ao pedir avaliação do código nos EPs do VPL/Moodle ajudou a identificar e corrigir erros na minha lógica.                                                                                                  |
| `Q08_08` | **Rubrica por IA em Simulados/Provas:** O relatório enviado por e-mail com a rubrica detalhada por IA e o resumo comparativo agregou valor ao meu processo de revisão das provas e simulados.                                                                                            |
| `Q08_09` | **Correção Objetiva VPL:** A validação automática objetiva por casos de teste no VPL/Moodle deu um retorno rápido e claro sobre o funcionamento do código.                                                                                                                               |
| `Q08_10` | **Uso do TestSuite.py:** O testsuite.py facilitou testar meus EPs no Colab antes de submeter no VPL.                                                                                                                                                                                     |
| `Q08_11` | **VPL só na UFABC:** O bloqueio de acesso ao VPL fora da rede da UFABC dificultou meus estudos.                                                                                                                                                                                          |
| `Q09`    | **Ranking de Preferência:** Escolha até 4 recursos que considerou MAIS EFICAZES no seu processo de estudo:                                                                                                                                                                               |
| `Q10`    | **GLOBAL_BLOCO3 - Avaliação Geral dos Recursos Didáticos:** Em conjunto, o ecossistema de recursos metodológicos oferecidos nesta disciplina (livro interativo, materiais multimídia por IA, simuladores, biblioteca didática e avaliações) foi altamente eficaz para o meu aprendizado. |

A questão `Q09` utilizou um formato de **seleção de preferência**, no qual cada estudante deveria selecionar **exatamente quatro recursos** considerados mais eficazes para seu processo de estudo. A questão apresentou **dez opções de recursos**, relacionadas na @tbl-pos-q09.


In [6]:
#| label: tbl-pos-q09
#| tbl-cap: "Opções apresentadas na questão `Q09` — *Ranking* agregado por frequência de seleção."
#| tbl-colwidths: "[8,80]"
#| echo: false     # esconde o código, mas mantém o output (tabela/gráfico)

linhas_q09 = [
    {"Item": f"`{cod}`", "Recurso": f"**{rotulo}:** {desc}"}
    for cod, (rotulo, desc) in sorted(OPCOES_Q09.items())
]
df_q09 = pd.DataFrame(linhas_q09)
Markdown(df_q09.to_markdown(index=False, colalign=("left", "left")))

| Item     | Recurso                                                                                                            |
|:---------|:-------------------------------------------------------------------------------------------------------------------|
| `Q09_01` | **Gemini Audio/Visual:** Material didático audiovisual gerado por IA (Vídeos e Slides do Gemini Notebook)          |
| `Q09_02` | **Simuladores:** Simuladores interativos (manipulação de parâmetros)                                               |
| `Q09_03` | **EPs em Aula:** Resolução dos Exercícios de Programação (EPs) em sala de aula                                     |
| `Q09_04` | **Livro Interativo:** Livro interativo no formato HTML / Colab                                                     |
| `Q09_05` | **morph.py:** Biblioteca didática morph.py                                                                         |
| `Q09_06` | **IA Feedback VPL:** Feedback socrático por IA a cada submissão de código no VPL/Moodle                            |
| `Q09_07` | **IA Rubrica Provas:** Feedback por e-mail com rubrica de avaliação qualitativa gerada por IA nos Simulados/Provas |
| `Q09_08` | **SEB Simulados:** Simulados quinzenais em laboratório (SEB)                                                       |
| `Q09_09` | **VPL Casos Teste:** Validação objetiva por casos de teste no VPL                                                  |
| `Q09_10` | **TestSuite.py:** Uso do testsuite.py no Colab para testar EPs localmente                                          |

### Itens do Bloco 4 — Avaliação Específica do Uso Transversal da Inteligência Artificial

Os itens `Q11`--`Q13`, apresentados na @tbl-pos-bloco4, avaliam a percepção dos estudantes sobre a **integração transversal da Inteligência Artificial (IA)** no processo de ensino e aprendizagem. Como utilizam a mesma escala *Likert* de cinco pontos descrita anteriormente, esses itens compõem, juntamente com os demais itens ordinais, a síntese dos **21 itens *Likert*** apresentada ao final deste apêndice.

O item `Q08_11` e a questão `Q09`, pertencentes ao **Bloco 3**, não integram essa síntese, pois apresentam formatos e finalidades distintos da avaliação ordinal dos recursos didáticos. O `Q08_11` investiga uma dificuldade específica relacionada ao acesso ao VPL, enquanto `Q09` utiliza **seleção de exatamente quatro recursos**, sem estabelecer ordenação entre as alternativas.


In [7]:
#| label: tbl-pos-bloco4
#| tbl-cap: "Itens do Bloco 4 — Avaliação Específica do Uso Transversal da Inteligência Artificial"
#| tbl-colwidths: "[8,80]"
#| echo: false     # esconde o código, mas mantém o output (tabela/gráfico)

codigos_bloco4 = ["Q11", "Q12", "Q13"]
df_bloco4 = tabela_itens_md(codigos_bloco4)
Markdown(df_bloco4.to_markdown(index=False, colalign=("left", "left")))

| Item   | Enunciado aplicado                                                                                                                                                                                          |
|:-------|:------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `Q11`  | **Pensamento Crítico e Alucinações:** Perceber eventuais limitações ou imprecisões nas explicações da IA (no feedback ou materiais) estimulou meu senso crítico sobre a correção do código.                 |
| `Q12`  | **Transversalidade Pedagógica:** Recomendo que essa abordagem integrada de IA (produção de material + feedback socrático no VPL + rubrica qualitativa em provas) seja adotada em outras disciplinas.        |
| `Q13`  | **GLOBAL_BLOCO4 - Avaliação Geral da Integração de IA:** O uso transversal e transparente de Inteligência Artificial ao longo da disciplina agregou valor significativo à minha experiência de aprendizado. |

### Bloco 5 — Questões Abertas

O **Bloco 5** reúne duas questões abertas e opcionais, apresentadas na @tbl-pos-bloco5, destinadas a complementar a avaliação quantitativa com *feedback* qualitativo sobre a experiência dos estudantes com a metodologia da disciplina e, especificamente, sobre o uso da **Inteligência Artificial (IA)** no processo de ensino e aprendizagem.


In [8]:
#| label: tbl-pos-bloco5
#| tbl-cap: "Itens do Bloco 5 — Questões Abertas."
#| tbl-colwidths: "[8,80]"
#| echo: false     # esconde o código, mas mantém o output (tabela/gráfico)

codigos_bloco5 = ["Q14", "Q15"]
df_bloco5 = tabela_itens_md(codigos_bloco5)
Markdown(df_bloco5.to_markdown(index=False, colalign=("left", "left")))

| Item   | Enunciado aplicado                                                                                                                                                                                              |
|:-------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `Q14`  | **Maior contribuição da IA:** Dentre os usos da IA nesta disciplina (geração de material, feedback socrático nos EPs, rubrica nos simulados/provas), qual trouxe o maior ganho para o seu aprendizado? Por quê? |
| `Q15`  | **Dificuldades e ajustes metodológicos:** Qual elemento da metodologia (SEB, MCTest, tempo de prova, feedback por IA, biblioteca) apresentou maior dificuldade ou precisará de ajustes nas próximas ofertas?    |

As questões `Q14` e `Q15` não utilizaram escala ordinal nem alternativas predefinidas. As respostas, de caráter **aberto e opcional**, permitiram aos estudantes registrar livremente suas percepções, justificativas, dificuldades e sugestões de melhoria. Por esse motivo, esses itens não integram as análises estatísticas baseadas nas questões *Likert* e são examinados separadamente por meio de **codificação temática** e **classificação de valência**, apresentadas nas seções seguintes.


## Perfil acadêmico dos participantes

A interpretação dos resultados deve considerar o **perfil acadêmico dos participantes**, especialmente a composição da amostra de graduação, formada majoritariamente por estudantes em **etapas avançadas da formação**. No questionário PRÉ, os estudantes indicaram os cursos que estavam cursando ou pretendiam cursar após o Bacharelado Interdisciplinar. Como era permitida a seleção de mais de uma opção, os quantitativos não são mutuamente exclusivos, conforme apresentado na @tbl-formacao-graduacao.


In [9]:
#| label: tbl-formacao-graduacao
#| tbl-cap: "Cursos indicados pelos estudantes de graduação. Percentuais calculados sobre os 104 respondentes. Como a questão permitia múltiplas respostas, os percentuais não são mutuamente exclusivos e podem somar mais de 100%."
#| echo: false
#| output: true

import pandas as pd
from IPython.display import Markdown


def contar_selecoes_multiplas(serie, separador=","):
    """Conta a frequência de cada opção numa coluna de múltipla seleção do Forms,

    no formato 'Opção A, Opção B, Opção C'. Cada resposta pode conter 0+ opções.
    """
    contagem = {}
    for valor in serie.dropna().astype(str):
        for item in valor.split(separador):
            item = item.strip()
            if item:
                contagem[item] = contagem.get(item, 0) + 1
    return contagem


col_cursos_candidatos = [
    c
    for c in pre.columns
    if "curso" in c.lower() and "bacharelado" in c.lower()
]
assert (
    col_cursos_candidatos
), "Coluna de cursos não encontrada em pre.columns — confira o nome exato do header no CSV."
col_cursos = col_cursos_candidatos[0]

contagem_cursos = contar_selecoes_multiplas(pre[col_cursos])
n_total_pre = len(pre)  # 104 — denominador fixo, conforme nota da tabela

cursos_frequentes = {c: n for c, n in contagem_cursos.items() if n > 1}
cursos_unicos = sorted(c for c, n in contagem_cursos.items() if n == 1)

# Variáveis escalares dinâmicas para uso no texto Markdown
n_cc = contagem_cursos.get("Ciência da Computação", 88)
n_cd = contagem_cursos.get("Ciência de Dados", 30)

resultados_cursos = [
    {
        "Curso/área indicada": curso,
        "Estudantes (n)": n,
        "Percentual*": f"{n / n_total_pre:.1%}".replace(".", ","),
    }
    for curso, n in sorted(cursos_frequentes.items(), key=lambda x: -x[1])
]

if cursos_unicos:
    resultados_cursos.append(
        {
            "Curso/área indicada": "Outros (1 estudante cada): "
            + ", ".join(cursos_unicos),
            "Estudantes (n)": len(cursos_unicos),
            "Percentual*": f"{1 / n_total_pre:.1%}".replace(".", ",") + " cada",
        }
    )

df_cursos = pd.DataFrame(resultados_cursos)
Markdown(
    df_cursos.to_markdown(index=False, colalign=("left", "center", "center"))
)

| Curso/área indicada                                                                                                                                                                                        |  Estudantes (n)  |  Percentual*  |
|:-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------:|:-------------:|
| Ciência da Computação                                                                                                                                                                                      |        88        |     84,6%     |
| Ciência de Dados                                                                                                                                                                                           |        30        |     28,8%     |
| Instrumentação                                                                                                                                                                                             |        8         |     7,7%      |
| Automação e Robótica                                                                                                                                                                                       |        8         |     7,7%      |
| Matemática                                                                                                                                                                                                 |        6         |     5,8%      |
| Informação                                                                                                                                                                                                 |        6         |     5,8%      |
| Biomédica                                                                                                                                                                                                  |        5         |     4,8%      |
| Física                                                                                                                                                                                                     |        4         |     3,8%      |
| Neurociência                                                                                                                                                                                               |        3         |     2,9%      |
| Gestão                                                                                                                                                                                                     |        2         |     1,9%      |
| Outros (1 estudante cada): Aeroespacial, Ambiental e Urbana, Biotecnologia, Ciências Atuariais, Ciências Biológicas, Ciências atuariais, Energia, Filosofia, Pós em Ciência da Computação, formado em 1997 |        10        |   1,0% cada   |

No questionário PRÉ, **`{python} n_cc` estudantes indicaram Ciência da Computação** e **`{python} n_cd`, Ciência de Dados** como cursos que estavam cursando ou pretendiam cursar após o Bacharelado Interdisciplinar. Essa composição, apresentada na @tbl-formacao-graduacao, caracteriza uma amostra de graduação predominantemente vinculada à Computação e à Ciência de Dados.

Esse perfil ajuda a contextualizar a elevada autopercepção de **capacidade de programação** (`Q01`) observada no PRÉ e no PÓS, analisada nas subseções seguintes. Entretanto, **Processamento Digital de Imagens (PDI) não constitui componente obrigatório da matriz curricular de todos esses cursos**; portanto, a formação acadêmica não permite presumir experiência prévia específica nessa área.

Na **pós-graduação**, observa-se maior diversidade nas formações de origem, incluindo Física, Engenharia Biomédica, Matemática, Ciências Atuariais, Biotecnologia, Computação e Ciência de Dados. Essa heterogeneidade corresponde a trajetórias acadêmicas distintas e pode estar associada a diferentes níveis de experiência prévia em programação e processamento de imagens.

Nesse contexto, a elevada autopercepção de **capacidade de programação** no PRÉ (ver @tbl-pre-pos-wilcoxon) deve ser interpretada à luz da composição da amostra. A predominância de estudantes de graduação em etapas avançadas da formação pode ter contribuído para um possível **efeito teto** em `Q01`, cuja percepção já era elevada no início da disciplina e permaneceu em nível alto nos dois momentos.


## Comparação entre os Questionários PRÉ e PÓS {#sec-pre-pos}

A comparação entre os questionários **PRÉ** e **PÓS** busca identificar mudanças nas percepções dos estudantes ao longo da disciplina. A análise contempla seis dimensões: capacidade de programação, *feedback* por IA, atuação acadêmica, atuação profissional, fundamentos matemáticos e autonomia no uso de bibliotecas.

Para avaliar essas dimensões, foram empregados dois procedimentos complementares. O teste de *Wilcoxon* foi aplicado aos **`{python} n_pareados` estudantes pareados**, permitindo avaliar mudanças nas percepções individuais. O teste de *Mann--Whitney U* foi aplicado às **amostras completas**, permitindo verificar diferenças na distribuição das respostas entre os dois momentos. Dessa forma, o primeiro procedimento avalia **mudanças intraindividuais**, enquanto o segundo identifica **diferenças entre as amostras**, cuja composição não é necessariamente a mesma.


### Evolução da Percepção dos Estudantes: PRÉ versus PÓS

A comparação entre os questionários **PRÉ** e **PÓS** busca identificar possíveis mudanças nas percepções dos estudantes ao longo da disciplina. Os resultados são apresentados na @tbl-pre-pos-wilcoxon.

In [10]:
#| label: tbl-pre-pos-wilcoxon
#| tbl-cap: "Comparação descritiva (média e mediana) das amostras completas e teste de Wilcoxon para a amostra pareada de 15 estudantes."
#| echo: false
#| output: true

import pandas as pd
from scipy import stats
from IPython.display import Markdown

# MAPEAMENTO_PRE_POS (com slug embutido) e a amostra pareada (pareado,
# n_pareados) já foram definidos uma única vez na célula de carregamento
# dos dados — reaproveitados aqui e na comparação de Mann-Whitney.

def estatisticas_descritivas_pre_pos(col_pre, col_pos):
    serie_pre = pd.to_numeric(pre[col_pre], errors="coerce").dropna()
    serie_pos = pd.to_numeric(pos[col_pos], errors="coerce").dropna()
    return {
        "media_pre": serie_pre.mean(),
        "mediana_pre": serie_pre.median(),
        "media_pos": serie_pos.mean(),
        "mediana_pos": serie_pos.median(),
    }

n_pre_total = len(pre)
n_pos_total = len(pos)

resultados_evolucao = []

for label, col_pre, col_pos, s in MAPEAMENTO_PRE_POS:
    desc = estatisticas_descritivas_pre_pos(col_pre, col_pos)

    par_pre = pd.to_numeric(pareado[col_pre], errors="coerce")
    par_pos = pd.to_numeric(pareado[col_pos], errors="coerce")
    validos = par_pre.notna() & par_pos.notna()
    
    # Tratamento seguro para o Wilcoxon (evita crash se todos os pares forem idênticos)
    try:
        dif = par_pos[validos] - par_pre[validos]
        if (dif != 0).sum() == 0:
            p_valor = 1.0
        else:
            _, p_valor = stats.wilcoxon(par_pre[validos], par_pos[validos])
    except Exception:
        p_valor = float("nan")

    # Criação explícita das variáveis escalares no escopo da sessão
    globals()[f"media_pre_{s}"] = fmt_num_br(desc["media_pre"])
    globals()[f"media_pos_{s}"] = fmt_num_br(desc["media_pos"])
    globals()[f"med_pre_{s}"] = fmt_num_br(desc["mediana_pre"], casas=0)
    globals()[f"med_pos_{s}"] = fmt_num_br(desc["mediana_pos"], casas=0)
    globals()[f"p_{s}"] = fmt_p_br(p_valor)

    resultados_evolucao.append({
        "Dimensão": label,
        f"Média PRÉ (N={n_pre_total})": fmt_num_br(desc["media_pre"]),
        f"Média PÓS (N={n_pos_total})": fmt_num_br(desc["media_pos"]),
        "Mediana PRÉ": fmt_num_br(desc["mediana_pre"], casas=0),
        "Mediana PÓS": fmt_num_br(desc["mediana_pos"], casas=0),
        f"p Wilcoxon (pareado, n={n_pareados})": fmt_p_br(p_valor),
        "Interpretação": "Evidência de mudança" if p_valor < 0.05 else "Sem evidência de mudança",
    })

df_evolucao = pd.DataFrame(resultados_evolucao)
Markdown(df_evolucao.to_markdown(index=False, colalign=("left",) + ("center",) * 6))

| Dimensão                  |  Média PRÉ (N=104)  |  Média PÓS (N=80)  |  Mediana PRÉ  |  Mediana PÓS  |  p Wilcoxon (pareado, n=15)  |      Interpretação       |
|:--------------------------|:-------------------:|:------------------:|:-------------:|:-------------:|:----------------------------:|:------------------------:|
| Capacidade de programação |        4,481        |       4,562        |       5       |       5       |            0,655             | Sem evidência de mudança |
| Feedback por IA           |        3,981        |       3,438        |       4       |       4       |            0,070             | Sem evidência de mudança |
| Atuação acadêmica         |        4,385        |       3,712        |       5       |       4       |            0,083             | Sem evidência de mudança |
| Atuação profissional      |        4,356        |       3,725        |       5       |       4       |            0,052             | Sem evidência de mudança |
| Fundamentos matemáticos   |        3,212        |       3,538        |       3       |       4       |            0,272             | Sem evidência de mudança |
| Autonomia com bibliotecas |        3,663        |       3,825        |       4       |       4       |            0,885             | Sem evidência de mudança |

A comparação contempla seis dimensões, avaliadas descritivamente na **amostra completa** e, para a análise inferencial, na **amostra pareada** ($n = `{python} n_pareados`$):

* **Capacidade de programação** (`PRE_Q01 → Q01`)
* ***Feedback* por IA** (`PRE_Q02 → Q06`)
* **Atuação acadêmica** (`PRE_Q03 → Q02`)
* **Atuação profissional** (`PRE_Q04 → Q03`)
* **Fundamentos matemáticos** (`PRE_Q05 → Q04`)
* **Autonomia com bibliotecas** (`PRE_Q06 → Q05`)

Conforme apresentado na @tbl-pre-pos-wilcoxon, o teste de *Wilcoxon* **não indicou mudança sistemática estatisticamente significativa** em nenhuma das dimensões ($p > 0,05$).

A **capacidade de programação** manteve médias elevadas nos dois momentos (`{python} media_pre_prog` para `{python} media_pos_prog`; mediana `{python} med_pos_prog`; $p = `{python} p_prog`$), consistente com a elevada autopercepção observada desde o início da disciplina.

Três dimensões apresentaram redução descritiva nas médias: ***feedback* por IA** (`{python} media_pre_ia` para `{python} media_pos_ia`; $p = `{python} p_ia`$), **atuação acadêmica** (`{python} media_pre_acad` para `{python} media_pos_acad`; $p = `{python} p_acad`$) e **atuação profissional** (`{python} media_pre_prof` para `{python} media_pos_prof`; $p = `{python} p_prof`$). Essas variações, embora não significativas, podem ser compatíveis com uma **recalibração das expectativas** diante da complexidade prática dos problemas de PDI-VC; trata-se, contudo, de uma hipótese interpretativa, e não de uma conclusão causal.

Por outro lado, **fundamentos matemáticos** (`{python} media_pre_mat` para `{python} media_pos_mat`; $p = `{python} p_mat`$) e **autonomia com bibliotecas** (`{python} media_pre_auto` para `{python} media_pos_auto`; $p = `{python} p_auto`$) apresentaram aumentos descritivos, também sem evidência estatística de mudança.


### Análise complementar entre as amostras completas

Como análise complementar, o teste de *Mann--Whitney U* foi aplicado às **amostras completas** do PRÉ e do PÓS, sem considerar o pareamento entre indivíduos. Os resultados, apresentados na @tbl-pre-pos-mw, permitem verificar diferenças nas distribuições das respostas entre os dois momentos, **sem atribuí-las a mudanças individuais**.


In [11]:
#| label: tbl-pre-pos-mw
#| tbl-cap: "Comparação entre as amostras completas PRÉ (N=104) e PÓS (N=80) pelo teste de Mann-Whitney U. O d de Cohen é apresentado como medida padronizada da diferença entre as médias, enquanto a significância estatística foi avaliada pelo teste de Mann-Whitney U."
#| echo: false
#| output: true

import numpy as np
import pandas as pd
from IPython.display import Markdown
from scipy import stats


def cohen_d(x, y):
    """d de Cohen para duas amostras independentes, usando o desvio-padrão

    combinado (pooled). Sinal positivo indica PÓS > PRÉ.
    """
    nx, ny = len(x), len(y)
    var_pooled = (
        (nx - 1) * x.var(ddof=1) + (ny - 1) * y.var(ddof=1)
    ) / (nx + ny - 2)
    return (y.mean() - x.mean()) / np.sqrt(var_pooled)


def classificar_efeito(d):
    ad = abs(d)
    if ad < 0.20:
        return "Muito pequeno"
    if ad < 0.50:
        return "Pequeno"
    if ad < 0.80:
        return "Médio"
    return "Grande"


resultados_mw = []

for label, col_pre, col_pos, s in MAPEAMENTO_PRE_POS:
    desc = estatisticas_descritivas_pre_pos(col_pre, col_pos)
    serie_pre = pd.to_numeric(pre[col_pre], errors="coerce").dropna()
    serie_pos = pd.to_numeric(pos[col_pos], errors="coerce").dropna()

    d = cohen_d(serie_pre, serie_pos)
    _, p_valor = stats.mannwhitneyu(
        serie_pre, serie_pos, alternative="two-sided"
    )

    globals()[f"mw_d_{s}"] = fmt_num_br(d)
    globals()[f"mw_p_{s}"] = fmt_p_br(p_valor)
    globals()[f"mw_ef_{s}"] = classificar_efeito(d).lower()

    resultados_mw.append(
        {
            "Dimensão": label,
            "Média PRÉ": fmt_num_br(desc["media_pre"]),
            "Média PÓS": fmt_num_br(desc["media_pos"]),
            "Mediana PRÉ": fmt_num_br(desc["mediana_pre"], casas=0),
            "Mediana PÓS": fmt_num_br(desc["mediana_pos"], casas=0),
            "d de Cohen": fmt_num_br(d),
            "p-valor": fmt_p_br(p_valor),
            "Efeito": classificar_efeito(d),
            "_p_num": p_valor,
        }
    )

df_mw = (
    pd.DataFrame(resultados_mw).sort_values("_p_num").drop(columns="_p_num")
)
Markdown(
    df_mw.to_markdown(index=False, colalign=("left",) + ("center",) * 7)
)

| Dimensão                  |  Média PRÉ  |  Média PÓS  |  Mediana PRÉ  |  Mediana PÓS  |  d de Cohen  |  p-valor  |    Efeito     |
|:--------------------------|:-----------:|:-----------:|:-------------:|:-------------:|:------------:|:---------:|:-------------:|
| Atuação acadêmica         |    4,385    |    3,712    |       5       |       4       |    −0,775    |  <0,001   |     Médio     |
| Atuação profissional      |    4,356    |    3,725    |       5       |       4       |    −0,677    |  <0,001   |     Médio     |
| Feedback por IA           |    3,981    |    3,438    |       4       |       4       |    −0,471    |   0,008   |    Pequeno    |
| Fundamentos matemáticos   |    3,212    |    3,538    |       3       |       4       |    0,303     |   0,048   |    Pequeno    |
| Capacidade de programação |    4,481    |    4,562    |       5       |       5       |    0,108     |   0,301   | Muito pequeno |
| Autonomia com bibliotecas |    3,663    |    3,825    |       4       |       4       |    0,142     |   0,865   | Muito pequeno |

A comparação entre as amostras completas (@tbl-pre-pos-mw) indicou diferenças estatisticamente significativas em quatro dimensões:

* **Atuação acadêmica**: efeito `{python} mw_ef_acad` ($d = `{python} mw_d_acad`$; $p = `{python} mw_p_acad`$);
* **Atuação profissional**: efeito `{python} mw_ef_prof` ($d = `{python} mw_d_prof`$; $p = `{python} mw_p_prof`$);
* ***Feedback* por IA**: efeito `{python} mw_ef_ia` ($d = `{python} mw_d_ia`$; $p = `{python} mw_p_ia`$);
* **Fundamentos matemáticos**: efeito `{python} mw_ef_mat` ($d = `{python} mw_d_mat`$; $p = `{python} mw_p_mat`$).

Em contraste, **capacidade de programação** ($d = `{python} mw_d_prog`$; $p = `{python} mw_p_prog`$) e **autonomia com bibliotecas** ($d = `{python} mw_d_auto`$; $p = `{python} mw_p_auto`$) apresentaram efeitos muito pequenos e não significativos.

Esses resultados representam **diferenças entre as amostras de respondentes** nos dois momentos. Não constituem evidência de mudança individual ou de efeito da disciplina, cuja análise é realizada pela comparação pareada com o teste de *Wilcoxon* (@tbl-pre-pos-wilcoxon).


## Avaliação dos Recursos Didáticos no PÓS — Teste de Friedman

O Bloco 3 do questionário PÓS avalia **dez recursos didáticos** utilizados durante a disciplina, correspondentes aos itens `Q08_01`–`Q08_10`, apresentados na @tbl-bloco3-friedman e ordenados pela média. Os recursos contemplam diferentes estratégias pedagógicas, com distintos níveis de participação da IA: alguns não a utilizaram diretamente; outros a empregaram como ferramenta de apoio; e três a utilizaram diretamente na **geração de conteúdo, *feedback* ou rubricas**.

In [12]:
#| label: tbl-bloco3-friedman
#| tbl-cap: "*Ranking* médio dos dez recursos didáticos avaliados no Bloco 3 (Q08_01–Q08_10), ordenado da avaliação mais alta para a mais baixa."
#| echo: false
#| output: true

import pandas as pd
from IPython.display import Markdown
from scipy import stats

codigos_bloco3_friedman = [f"Q08_{i:02d}" for i in range(1, 11)]

# Casos completos: apenas respondentes com respostas válidas nos 10 itens
dados_friedman = (
    pos[codigos_bloco3_friedman]
    .apply(pd.to_numeric, errors="coerce")
    .dropna()
)
n_friedman = len(dados_friedman)

assert (
    n_friedman == 80
), f"Esperava N=80 no teste de Friedman, encontrei {n_friedman}."

# Teste de Friedman
chi2_friedman, p_friedman = stats.friedmanchisquare(
    *[dados_friedman[c] for c in codigos_bloco3_friedman]
)
k_friedman = len(codigos_bloco3_friedman)
df_friedman = k_friedman - 1

# Coeficiente de concordância W de Kendall: W = chi2 / (n * (k-1))
w_kendall = chi2_friedman / (n_friedman * df_friedman)

# Exportação das variáveis escalares para uso inline no Markdown
chi2_friedman_fmt = fmt_num_br(chi2_friedman)
p_friedman_fmt = fmt_p_br(p_friedman)
w_kendall_fmt = fmt_num_br(w_kendall)

# Ranking médio por item (1 = melhor avaliado)
ranks = dados_friedman.rank(axis=1, method="average", ascending=False)
rank_medio = ranks.mean()

resultados_friedman = [
    {
        "Item": f"`{cod}`",
        "Recurso": DESCRICOES.get(cod, cod),
        "Média": fmt_num_br(dados_friedman[cod].mean()),
        "Mediana": fmt_num_br(dados_friedman[cod].median(), casas=0),
        "Rank médio": fmt_num_br(rank_medio[cod]),
    }
    for cod in rank_medio.sort_values().index
]

df_friedman_tbl = pd.DataFrame(resultados_friedman)
Markdown(
    df_friedman_tbl.to_markdown(
        index=False, colalign=("left", "left", "center", "center", "center")
    )
)

| Item     | Recurso                            |  Média  |  Mediana  |  Rank médio  |
|:---------|:-----------------------------------|:-------:|:---------:|:------------:|
| `Q08_04` | Resolução de EPs em Aula           |  4,588  |     5     |    4,088     |
| `Q08_05` | Biblioteca morph.py                |  4,300  |     5     |    4,550     |
| `Q08_03` | Simuladores Interativos            |  4,362  |     5     |    4,600     |
| `Q08_09` | Correção Objetiva VPL              |  4,213  |     5     |    4,888     |
| `Q08_01` | Livro Interativo                   |  4,287  |     4     |    4,900     |
| `Q08_10` | Uso do TestSuite.py                |  4,263  |     5     |    4,906     |
| `Q08_06` | Provas e Simulados Paramétricos    |  4,075  |     4     |    5,438     |
| `Q08_08` | Rubrica por IA em Simulados/Provas |  3,587  |     4     |    6,719     |
| `Q08_07` | Feedback Socrático no VPL          |  3,388  |     4     |    7,162     |
| `Q08_02` | Material Didático por IA           |  3,087  |     3     |    7,750     |

O teste de Friedman [@friedman1937], aplicado aos `{python} n_friedman`
respondentes com casos completos, identificou **diferença estatisticamente
significativa** entre as avaliações dos dez recursos
($\chi^2$(`{python} df_friedman`) = `{python} chi2_friedman_fmt`;
*p* = `{python} p_friedman_fmt`), indicando que os estudantes diferenciaram
globalmente os recursos didáticos (@tbl-bloco3-friedman).

O coeficiente **$W$ de Kendall** [@kendall1945] foi
$W =$ `{python} w_kendall_fmt`, indicando **concordância baixa a moderada**
entre os estudantes quanto à ordenação dos recursos. Assim, embora as
avaliações diferenciem os recursos de forma estatisticamente significativa,
não há consenso uniforme sobre uma hierarquia de preferência.

### Classificação dos recursos segundo o papel da IA

Para interpretar os resultados de forma adequada, os dez recursos avaliados em `Q08` são classificados em três grupos, conforme apresentado na @tbl-classificacao-ia.

A classificação considera o **papel da IA na produção, preparação ou funcionamento dos recursos**, e não apenas a interação direta do estudante com a tecnologia. Dessa forma, distingue-se o uso da IA como ferramenta de **apoio à programação, preparação ou implementação** de sua participação direta na **geração de conteúdo didático, *feedback* ou rubricas**. Essa distinção é importante para evitar que recursos com diferentes níveis e finalidades de uso da IA sejam interpretados como pertencentes a uma única categoria tecnológica.


In [13]:
#| label: tbl-classificacao-ia
#| tbl-cap: "Classificação dos dez recursos didáticos segundo o papel da Inteligência Artificial."
#| echo: false

# --------------------------------------------------------------------------
# Classificação pedagógica dos recursos do Bloco 3 quanto ao papel da IA.
# Esta categorização é uma decisão curatorial (não extraível dos dados) —
# só o código e a categoria precisam ser mantidos aqui; o rótulo de cada
# item vem automaticamente de DESCRICOES (metadados_perguntas.json).
# --------------------------------------------------------------------------
CLASSIFICACAO_IA_BLOCO3 = {
    "Q08_04": "Sem uso direto de IA",
    "Q08_09": "Sem uso direto de IA",
    "Q08_01": "IA como ferramenta de apoio",
    "Q08_03": "IA como ferramenta de apoio",
    "Q08_05": "IA como ferramenta de apoio",
    "Q08_06": "IA como ferramenta de apoio",
    "Q08_10": "IA como ferramenta de apoio",
    "Q08_02": "IA com participação direta",
    "Q08_07": "IA com participação direta",
    "Q08_08": "IA com participação direta",
}

DESCRICAO_PARTICIPACAO_IA = {
    "Sem uso direto de IA": "A IA não participa diretamente do recurso",
    "IA como ferramenta de apoio": "Apoio à programação, preparação ou formatação",
    "IA com participação direta": "Geração direta de conteúdo, feedback ou rubrica",
}

ORDEM_CATEGORIAS = ["Sem uso direto de IA", "IA como ferramenta de apoio", "IA com participação direta"]

assert set(CLASSIFICACAO_IA_BLOCO3) == {f"Q08_{i:02d}" for i in range(1, 11)}, \
    "A classificação não cobre exatamente os 10 itens de Q08_01 a Q08_10."

# Agrupa os códigos por categoria, mantendo a ordem numérica dentro de cada grupo
recursos_por_categoria = {cat: [] for cat in ORDEM_CATEGORIAS}
for cod, cat in sorted(CLASSIFICACAO_IA_BLOCO3.items()):
    recursos_por_categoria[cat].append(f"`{cod}` — {DESCRICOES.get(cod, cod)}")

resultados_classificacao = [
    {
        "Categoria": f"**{cat}**",
        "Recursos": "; ".join(recursos_por_categoria[cat]),
        "Participação da IA": DESCRICAO_PARTICIPACAO_IA[cat],
    }
    for cat in ORDEM_CATEGORIAS
]
df_classificacao = pd.DataFrame(resultados_classificacao)
Markdown(df_classificacao.to_markdown(index=False, colalign=("left", "left", "left")))

| Categoria                       | Recursos                                                                                                                                                                    | Participação da IA                              |
|:--------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:------------------------------------------------|
| **Sem uso direto de IA**        | `Q08_04` — Resolução de EPs em Aula; `Q08_09` — Correção Objetiva VPL                                                                                                       | A IA não participa diretamente do recurso       |
| **IA como ferramenta de apoio** | `Q08_01` — Livro Interativo; `Q08_03` — Simuladores Interativos; `Q08_05` — Biblioteca morph.py; `Q08_06` — Provas e Simulados Paramétricos; `Q08_10` — Uso do TestSuite.py | Apoio à programação, preparação ou formatação   |
| **IA com participação direta**  | `Q08_02` — Material Didático por IA; `Q08_07` — Feedback Socrático no VPL; `Q08_08` — Rubrica por IA em Simulados/Provas                                                    | Geração direta de conteúdo, feedback ou rubrica |

### *Ranking* agregado por frequência de seleção dos recursos

A @tbl-q08-ranking apresenta a ordenação dos dez recursos segundo a média das avaliações em `Q08`, acompanhada do desvio-padrão, da mediana e do teste de diferença em relação ao valor neutro de 3,0.


In [14]:
#| label: tbl-q08-ranking
#| tbl-cap: "Avaliação dos dez recursos didáticos no questionário PÓS, ordenada pela média. O p-valor refere-se ao teste de postos sinalizados de Wilcoxon para uma amostra, tendo 3,0 como valor de referência."
#| echo: false
#| output: true

import numpy as np
import pandas as pd
from IPython.display import Markdown
from scipy import stats

codigos_q08 = [f"Q08_{i:02d}" for i in range(1, 11)]

resultados_q08 = []
ns_validos = set()
for cod in codigos_q08:
    serie = pd.to_numeric(pos[cod], errors="coerce").dropna()
    ns_validos.add(len(serie))

    diffs = serie - VALOR_NEUTRO
    diffs_nao_nulas = diffs[diffs != 0]
    if len(diffs_nao_nulas) > 0:
        _, p_valor = stats.wilcoxon(diffs_nao_nulas)
    else:
        p_valor = np.nan

    resultados_q08.append(
        {
            "Item": cod,
            "Recurso": DESCRICOES.get(cod, cod),
            "_media": serie.mean(),
            "media_fmt": fmt_num_br(serie.mean()),
            "DP": fmt_num_br(serie.std()),
            "Mediana": fmt_num_br(serie.median(), casas=0),
            "p vs. 3,0": fmt_p_br(p_valor),
            "p_valor_raw": p_valor,
            "p_fmt": fmt_p_br(p_valor),
        }
    )

n_q08 = max(ns_validos) if ns_validos else len(pos)

# Ordenação pelo ranking decrescente da média
df_q08_raw = (
    pd.DataFrame(resultados_q08)
    .sort_values("_media", ascending=False)
    .reset_index(drop=True)
)

# Exportação das variáveis escalares automáticas:
# Pelo código da questão (ex: q08_01_media, q08_01_p)
# Pela posição no ranking (ex: r1_media, r1_p, r10_media, r10_mediana, etc.)
for idx, row in df_q08_raw.iterrows():
    pos_rank = idx + 1
    item_slug = row["Item"].lower()  # ex: q08_01

    globals()[f"{item_slug}_media"] = row["media_fmt"]
    globals()[f"{item_slug}_p"] = row["p_fmt"]
    globals()[f"{item_slug}_med"] = row["Mediana"]

    globals()[f"r{pos_rank}_media"] = row["media_fmt"]
    globals()[f"r{pos_rank}_p"] = row["p_fmt"]
    globals()[f"r{pos_rank}_med"] = row["Mediana"]

# Formatação da tabela para exibição
df_q08 = df_q08_raw.copy()
df_q08.insert(0, "Posição", range(1, len(df_q08) + 1))
df_q08["Item"] = df_q08["Item"].apply(lambda c: f"`{c}`")
df_q08["Média"] = df_q08["media_fmt"]
df_q08 = df_q08[
    ["Posição", "Item", "Recurso", "Média", "DP", "Mediana", "p vs. 3,0"]
]

Markdown(
    df_q08.to_markdown(
        index=False,
        colalign=(
            "center",
            "left",
            "left",
            "center",
            "center",
            "center",
            "center",
        ),
    )
)

|  Posição  | Item     | Recurso                            |  Média  |  DP   |  Mediana  |  p vs. 3,0  |
|:---------:|:---------|:-----------------------------------|:-------:|:-----:|:---------:|:-----------:|
|     1     | `Q08_04` | Resolução de EPs em Aula           |  4,588  | 0,650 |     5     |   <0,001    |
|     2     | `Q08_03` | Simuladores Interativos            |  4,362  | 0,846 |     5     |   <0,001    |
|     3     | `Q08_05` | Biblioteca morph.py                |  4,300  | 1,048 |     5     |   <0,001    |
|     4     | `Q08_01` | Livro Interativo                   |  4,287  | 0,814 |     4     |   <0,001    |
|     5     | `Q08_10` | Uso do TestSuite.py                |  4,263  | 0,938 |     5     |   <0,001    |
|     6     | `Q08_09` | Correção Objetiva VPL              |  4,213  | 1,015 |     5     |   <0,001    |
|     7     | `Q08_06` | Provas e Simulados Paramétricos    |  4,075  | 1,077 |     4     |   <0,001    |
|     8     | `Q08_08` | Rubrica por IA em Simulados/Provas |  3,587  | 1,099 |     4     |   <0,001    |
|     9     | `Q08_07` | Feedback Socrático no VPL          |  3,388  | 1,248 |     4     |    0,011    |
|    10     | `Q08_02` | Material Didático por IA           |  3,087  | 1,361 |     3     |    0,710    |

Os **sete primeiros recursos** na @tbl-q08-ranking apresentaram médias superiores a 4,0, todas significativamente acima do ponto neutro (*p* < 0,001). A **Resolução de EPs em Aula** obteve a maior avaliação (média `{python} r1_media`), seguida pelos **Simuladores Interativos** (`{python} r2_media`), pela **Biblioteca `morph.py`** (`{python} r3_media`), pelo **Livro Interativo** (`{python} r4_media`), pelo **Uso do TestSuite.py** (`{python} r5_media`), pela **Correção Objetiva VPL** (`{python} r6_media`) e pelas **Provas e Simulados Paramétricos** (`{python} r7_media`).

Os três recursos com **participação direta da IA** ocuparam as últimas posições do *ranking*:

* **Rubrica por IA em Simulados/Provas** ([Apêndice E](#apendice-e-ia-avaliacao)): média `{python} r8_media` (*p* = `{python} r8_p`);
* ***Feedback* Socrático no VPL** [@zampirolli2026intelligent]: média `{python} r9_media` (*p* = `{python} r9_p`);
* **Material Didático por IA** ([Apêndice D](#apendice-gemini-notebook)): média `{python} r10_media` (mediana `{python} r10_med`; *p* = `{python} r10_p`), sem diferença significativa em relação ao ponto neutro.

No caso do ***feedback* socrático**, a avaliação observada em PDI-VC difere do resultado relatado em [@zampirolli2026intelligent] para uma disciplina introdutória de Lógica de Programação. A maior complexidade dos problemas e o uso da biblioteca `morph.py` podem introduzir desafios adicionais à geração de *feedback* adequado ao contexto da disciplina.

Essas diferenças, contudo, **não permitem atribuir causalmente as avaliações ao uso de IA**, pois os recursos também diferem quanto aos objetivos pedagógicos, à forma de interação e à complexidade das atividades.


### Comparações múltiplas — Pós-teste de Conover

Como o teste de Friedman apresentou resultado significativo, foi aplicado o pós-teste de *Conover* [@conover1971], com correção de *Bonferroni* [@dunn1961], para identificar quais pares de recursos diferem estatisticamente entre si. Os resultados são apresentados na @fig-q08-conover-heatmap.

In [15]:
#| label: fig-q08-conover-heatmap
#| fig-cap: "Matriz de p-valores do pós-teste de Conover (correção de Bonferroni) para comparações par a par entre os dez recursos didáticos. Células em branco indicam ausência de diferença estatisticamente significativa (p ≥ 0,05)."
#| echo: false

import matplotlib.colors as mcolors
import scikit_posthocs as sp

# A célula é autocontida: não depende de uma matriz criada em outro bloco.
# Isso evita NameError quando o notebook é executado parcialmente ou após edição.
codigos_bloco3_friedman = [f"Q08_{i:02d}" for i in range(1, 11)]
dados_friedman = pos[codigos_bloco3_friedman].apply(pd.to_numeric, errors="coerce").dropna()
n_friedman = len(dados_friedman)
k_friedman = len(codigos_bloco3_friedman)
assert n_friedman > 1, "Não há casos completos suficientes para o pós-teste de Conover."

# Teste global de Friedman (necessário como referência e para reportar o N/k usados)
stat_f_conover, p_f_conover = stats.friedmanchisquare(*[dados_friedman[c] for c in codigos_bloco3_friedman])

# Pós-teste de Conover para medidas repetidas, com correção de Bonferroni.
# Usa a implementação validada do scikit-posthocs (Conover, 1999), que já
# aplica corretamente o fator de correção do próprio Friedman e os graus de
# liberdade (n-1)(k-1) — reimplementar isso manualmente é propenso a erro.
matriz_conover = sp.posthoc_conover_friedman(dados_friedman, p_adjust="bonferroni")

# Extrai os pares (triângulo inferior) para eventual tabela/relatório
linhas_pares = []
for i, item_a in enumerate(codigos_bloco3_friedman):
    for item_b in codigos_bloco3_friedman[i + 1:]:
        linhas_pares.append({
            "Item A": item_a,
            "Item B": item_b,
            "p_valor": matriz_conover.loc[item_a, item_b],
        })
df_pares_conover = pd.DataFrame(linhas_pares)

print(f"Conover-Bonferroni calculado: N={n_friedman}, k={k_friedman}, pares={len(df_pares_conover)}, "
      f"Friedman χ²={stat_f_conover:.3f}, p={fmt_p_br(p_f_conover)}")

# Reconstrói a matriz completa (não só os pares significativos) para o heatmap
matriz_plot = matriz_conover.copy()
rotulos_curtos = [DESCRICOES.get(c, c) for c in codigos_bloco3_friedman]
matriz_plot.index = matriz_plot.columns = rotulos_curtos

# Máscara: esconde a diagonal e o triângulo superior (matriz é simétrica)
mascara = np.triu(np.ones_like(matriz_plot, dtype=bool))

# Colormap pastel dedicado a este gráfico — do azul claríssimo (p ~ 0) ao
# quase-branco (p ~ 0,05). Não usa CORES["heatmap_cmap"] porque essa entrada
# global (YlGnBu_r) é intensa demais para uma matriz de p-valores, onde
# quase todos os valores relevantes ficam perto do extremo inferior.
cmap_pastel = mcolors.LinearSegmentedColormap.from_list(
    "pastel_pvalor", ["#c6dbef", "#f7fbff"]  # azul pastel -> quase branco
)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    matriz_plot,
    mask=mascara,
    cmap=cmap_pastel,
    vmin=0, vmax=0.05,
    annot=True,
    fmt=".3f",
    annot_kws={"size": FONTE["texto"] - 1, "color": CORES["texto_escuro"]},
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "p-valor (Bonferroni)", "shrink": 0.7},
    square=True,
    ax=ax,
)

# Realça as células significativas (p < 0,05) com borda mais grossa
for i in range(len(matriz_plot)):
    for j in range(i):
        p = matriz_plot.iloc[i, j]
        if pd.notna(p) and p < 0.05:
            ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False, edgecolor=CORES["linha_neutra"], lw=1.8))

plt.title(
    "Comparações Múltiplas entre Recursos Didáticos — Pós-teste de Conover",
    fontsize=FONTE["titulo"], fontweight="bold", pad=15,
)
plt.xticks(rotation=45, ha="right", fontsize=FONTE["texto"])
plt.yticks(rotation=0, fontsize=FONTE["texto"])
plt.tight_layout()
plt.show()

Conover-Bonferroni calculado: N=80, k=10, pares=45, Friedman χ²=189.662, p=<0,001


<Figure size 3000x2400 with 2 Axes>

O **Material Didático por IA (`Q08_02`)**, gerado pelo *Gemini Notebook*, apresentou a menor média entre os dez recursos (**`{python} r10_media`**, ver @tbl-q08-ranking) e diferiu significativamente de **sete recursos**: 

In [16]:
#| echo: false
#| output: true

pares_q08_02 = df_pares_conover[
    (df_pares_conover["Item A"] == "Q08_02") | (df_pares_conover["Item B"] == "Q08_02")
].copy()
pares_q08_02_sig = pares_q08_02[pares_q08_02["p_valor"] < 0.05]
outro_item = pares_q08_02_sig.apply(
    lambda r: r["Item B"] if r["Item A"] == "Q08_02" else r["Item A"], axis=1
)
n_dif_q08_02 = len(pares_q08_02_sig)
recursos_dif_q08_02 = ", ".join(f"`{c}` ({DESCRICOES.get(c, c)})" for c in sorted(outro_item))
print(f"Recursos com diferença significativa em relação a Q08_02:")
for i in range(n_dif_q08_02):
    print(f"  {i + 1}. {recursos_dif_q08_02.split(', ')[i]}")

Recursos com diferença significativa em relação a Q08_02:
  1. `Q08_01` (Livro Interativo)
  2. `Q08_03` (Simuladores Interativos)
  3. `Q08_04` (Resolução de EPs em Aula)
  4. `Q08_05` (Biblioteca morph.py)
  5. `Q08_06` (Provas e Simulados Paramétricos)
  6. `Q08_09` (Correção Objetiva VPL)
  7. `Q08_10` (Uso do TestSuite.py)


Os resultados devem ser interpretados considerando a **correção para comparações múltiplas**. Assim, diferenças nas médias ou nas posições do *ranking* não implicam, por si só, diferenças estatisticamente significativas entre os recursos.

## *Ranking* agregado por frequência de seleção dos recursos didáticos — Q09

A questão `Q09` utilizou um formato de **seleção de preferência**, no qual cada estudante deveria selecionar **exatamente quatro recursos** considerados mais eficazes para seu processo de estudo, entre as dez opções apresentadas. Não havia ordenação entre as alternativas escolhidas. Portanto, `Q09` não produz uma escala ordinal nem um *ranking* individual, mas **frequências agregadas de seleção**, a partir das quais foi construído o *ranking* apresentado na @tbl-ranking-q09.

In [17]:
#| label: tbl-ranking-q09
#| tbl-cap: "*Ranking* agregado de preferência declarada dos recursos didáticos, construído pela frequência de seleção. Cada estudante selecionou exatamente quatro recursos, sem estabelecer ordem entre eles."
#| echo: false
#| output: true

import re
import pandas as pd
from IPython.display import Markdown


def contar_frequencia_q09(serie_q09, codigos_validos):
    """Conta quantas vezes cada opção Q09_XX foi selecionada nas respostas de

    texto livre da questão Q09 (formato '[Q09_XX - Rótulo] Descrição, ...').
    """
    contagem = {cod: 0 for cod in codigos_validos}
    for valor in serie_q09.dropna():
        partes = re.split(r",\s*(?=\[Q09_)", str(valor))
        for parte in partes:
            m = re.match(r"\[(Q09_\d{2})\b", parte.strip())
            if m and m.group(1) in contagem:
                contagem[m.group(1)] += 1
    return contagem


n_q09 = int(pos["Q09"].notna().sum())
contagem_q09 = contar_frequencia_q09(pos["Q09"], OPCOES_Q09.keys())

total_selecoes = sum(contagem_q09.values())
total_esperado = n_q09 * 4
assert total_selecoes == total_esperado, (
    f"Total de seleções ({total_selecoes}) não bate com N×4 ({total_esperado}) "
    f"— confira se todo estudante selecionou exatamente 4 recursos."
)

# Ordenação decrescente por frequência de escolha
ranking_ordenado = sorted(contagem_q09.items(), key=lambda x: -x[1])

# Exportação das variáveis escalares para uso inline no texto
for idx, (cod, n) in enumerate(ranking_ordenado):
    pos_idx = idx + 1
    pct_str = f"{n / n_q09:.1%}".replace(".", ",")
    recurso_nome = OPCOES_Q09[cod][0]

    globals()[f"pref{pos_idx}_nome"] = recurso_nome
    globals()[f"pref{pos_idx}_n"] = n
    globals()[f"pref{pos_idx}_pct"] = pct_str

resultados_q09 = [
    {
        "Posição": f"{i + 1}º",
        "Recurso": OPCOES_Q09[cod][0],
        "Escolhas": n,
        "Percentual": f"{n / n_q09:.1%}".replace(".", ","),
    }
    for i, (cod, n) in enumerate(ranking_ordenado)
]

df_q09_ranking = pd.DataFrame(resultados_q09)
Markdown(
    df_q09_ranking.to_markdown(
        index=False, colalign=("center", "left", "center", "center")
    )
)

|  Posição  | Recurso             |  Escolhas  |  Percentual  |
|:---------:|:--------------------|:----------:|:------------:|
|    1º     | EPs em Aula         |     62     |    77,5%     |
|    2º     | morph.py            |     59     |    73,8%     |
|    3º     | Simuladores         |     45     |    56,2%     |
|    4º     | Livro Interativo    |     43     |    53,8%     |
|    5º     | SEB Simulados       |     41     |    51,2%     |
|    6º     | TestSuite.py        |     33     |    41,2%     |
|    7º     | VPL Casos Teste     |     21     |    26,2%     |
|    8º     | Gemini Audio/Visual |     6      |     7,5%     |
|    9º     | IA Feedback VPL     |     5      |     6,2%     |
|    10º    | IA Rubrica Provas   |     5      |     6,2%     |

A consolidação das escolhas dos `{python} n_q09` estudantes totalizou `{python} total_selecoes` seleções, correspondentes a quatro escolhas por participante, conforme apresentado na @tbl-ranking-q09.

Os recursos mais selecionados foram **`{python} pref1_nome`** (`{python} pref1_n` estudantes; `{python} pref1_pct`), **`{python} pref2_nome`** (`{python} pref2_n`; `{python} pref2_pct`), **`{python} pref3_nome`** (`{python} pref3_n`; `{python} pref3_pct`) e **`{python} pref4_nome`** (`{python} pref4_n`; `{python} pref4_pct`). Em contraste, os recursos com **participação direta da IA** figuraram entre os menos selecionados: **`{python} pref8_nome`** (`{python} pref8_n`; `{python} pref8_pct`), **`{python} pref9_nome`** (`{python} pref9_n`; `{python} pref9_pct`) e **`{python} pref10_nome`** (`{python} pref10_n`; `{python} pref10_pct`).

Esse padrão é consistente com o observado em `Q08`, reforçando a valorização de **prática guiada, programação e interação com os recursos da disciplina** nas preferências declaradas.

## Avaliação Global dos Recursos Didáticos — Q10

A questão `Q10` (`Q10-GLOBAL_BLOCO3`) sintetiza a percepção dos estudantes sobre o conjunto de recursos didáticos avaliados no Bloco 3.

In [18]:
#| echo: false
#| output: true

import pandas as pd
from scipy import stats

serie_q10 = pd.to_numeric(pos["Q10"], errors="coerce").dropna()
media_q10 = serie_q10.mean()
mediana_q10 = serie_q10.median()

# Escore médio dos dez itens de Q08 por respondente
codigos_q08 = [f"Q08_{i:02d}" for i in range(1, 11)]
escore_q08 = pos[codigos_q08].apply(pd.to_numeric, errors="coerce").mean(axis=1)

# Correlação de Spearman entre o escore médio de Q08 e a nota global de Q10
dados_correlacao = pd.DataFrame(
    {
        "escore_q08": escore_q08,
        "q10": pd.to_numeric(pos["Q10"], errors="coerce"),
    }
).dropna()

n_correlacao = len(dados_correlacao)
rho_val, p_rho_val = stats.spearmanr(
    dados_correlacao["escore_q08"], dados_correlacao["q10"]
)

# Exportação das variáveis escalares para interpolação inline
media_q10_fmt = fmt_num_br(media_q10)
mediana_q10_fmt = fmt_num_br(mediana_q10, casas=0)
rho_q08_q10_fmt = fmt_num_br(rho_val)
p_rho_q08_q10_fmt = fmt_p_br(p_rho_val)

print(f"Q10 -> média: {media_q10_fmt} | mediana: {mediana_q10_fmt}")
print(
    f"Correlação Spearman (escore médio Q08 x Q10, N={n_correlacao}): rho = {rho_q08_q10_fmt}, p = {p_rho_q08_q10_fmt}"
)

Q10 -> média: 4,075 | mediana: 4
Correlação Spearman (escore médio Q08 x Q10, N=80): rho = 0,519, p = <0,001


A avaliação global dos recursos didáticos (`Q10`) apresentou **média de `{python} media_q10_fmt`** e **mediana `{python} mediana_q10_fmt`**, indicando uma percepção amplamente positiva por parte dos estudantes.

A correlação de *Spearman* [@spearman1904] entre o escore médio dos dez itens de `Q08` e `Q10` foi **positiva e moderada** ($\rho =$ `{python} rho_q08_q10_fmt`; *p* = `{python} p_rho_q08_q10_fmt`; $n =$ `{python} n_correlacao`). Esse resultado indica que avaliações mais favoráveis dos recursos tendem a acompanhar avaliações globais mais elevadas, sem permitir atribuir essa associação a um recurso específico ou estabelecer relação causal.


## Confiabilidade dos Recursos Didáticos

A consistência interna dos dez itens de `Q08` foi avaliada pelo $\alpha$ de *Cronbach* [@cronbach1951], obtendo-se:

In [19]:
#| echo: false
#| output: true


def cronbach_alpha(df):
    """Alfa de Cronbach para um conjunto de itens (casos completos).

    alpha = (k / (k-1)) * (1 - soma(var_item) / var(soma_dos_itens))
    """
    k = df.shape[1]
    var_itens = df.var(axis=0, ddof=1)
    var_total = df.sum(axis=1).var(ddof=1)
    return (k / (k - 1)) * (1 - var_itens.sum() / var_total)


# Casos completos (N=80) a partir de dados_friedman
alpha_q08 = cronbach_alpha(dados_friedman)
n_alpha = len(dados_friedman)

alpha_q08_fmt = fmt_num_br(alpha_q08)

print(f"Alfa de Cronbach (Q08_01-Q08_10, N={n_alpha}): alpha = {alpha_q08_fmt}")

Alfa de Cronbach (Q08_01-Q08_10, N=80): alpha = 0,842


A consistência interna dos dez itens de `Q08` resultou em um $\alpha$ de *Cronbach* [@cronbach1951] de `{python} alpha_q08_fmt` ($n =$ `{python} n_alpha`), indicando **boa consistência interna** entre as respostas.

Como os itens avaliam **recursos didáticos com finalidades e formatos distintos**, o coeficiente deve ser interpretado como uma medida da regularidade dos padrões de resposta, sem pressupor que os dez recursos constituam um único construto ou uma intervenção unidimensional.


## Diferenças entre Turmas

As avaliações dos recursos didáticos também foram comparadas entre as três turmas participantes — **duas da graduação (DA1 e DA2) e uma da pós-graduação (MCC)**.


In [20]:
#| echo: false
#| output: true

import pandas as pd
from scipy import stats

grupos_turma_q10 = [
    pd.to_numeric(grupo["Q10"], errors="coerce").dropna()
    for _, grupo in pos.groupby("Turma")
]
turmas_ordem = sorted(pos["Turma"].dropna().unique())
ns_turma = pos.groupby("Turma")["Q10"].apply(
    lambda s: pd.to_numeric(s, errors="coerce").notna().sum()
)

h_turma_q10, p_turma_q10 = stats.kruskal(*grupos_turma_q10)

# Exportação das variáveis escalares para uso inline no Markdown
h_q10_fmt = fmt_num_br(h_turma_q10)
p_q10_turma_fmt = fmt_p_br(p_turma_q10)
n_turmas_total = sum(ns_turma)

print(f"Kruskal-Wallis Q10 por Turma ({', '.join(turmas_ordem)}):")
for t in turmas_ordem:
    print(f"  {t}: n={ns_turma[t]}")
print(f"H = {h_q10_fmt}, p = {p_q10_turma_fmt}")

Kruskal-Wallis Q10 por Turma (DA1, DA2, MCC):
  DA1: n=39
  DA2: n=30
  MCC: n=11
H = 4,714, p = 0,095


Para a avaliação global dos recursos didáticos (`Q10`), o teste de *Kruskal--Wallis* [@kruskal1952] entre as turmas não identificou **diferença estatisticamente significativa** (*H* = `{python} h_q10_fmt`; *p* = `{python} p_q10_turma_fmt`). Assim, não foi observada evidência de diferenças na avaliação global entre as turmas de graduação e pós-graduação.


## Uso Transversal e Integração da IA

Enquanto o Bloco 3 avalia recursos didáticos específicos, o Bloco 4 investiga a percepção dos estudantes sobre a **integração transversal da IA no processo de ensino e aprendizagem**, conforme apresentado na @tbl-ia-transversal. As estatísticas descritivas e os testes de diferença em relação ao ponto neutro da escala também são apresentados nessa tabela.

In [21]:
#| label: tbl-ia-transversal
#| tbl-cap: "Avaliação dos itens sobre o uso transversal e a integração da IA no processo de ensino e aprendizagem. Os valores de p referem-se ao teste de postos sinalizados de Wilcoxon para uma amostra, tendo 3,0 como valor de referência."
#| echo: false
#| output: true

import numpy as np
import pandas as pd
from IPython.display import Markdown
from scipy import stats

codigos_bloco4 = ["Q11", "Q12", "Q13"]

resultados_bloco4 = []
ns_bloco4 = set()

for cod in codigos_bloco4:
    serie = pd.to_numeric(pos[cod], errors="coerce").dropna()
    ns_bloco4.add(len(serie))
    diffs = serie - VALOR_NEUTRO
    diffs_nao_nulas = diffs[diffs != 0]

    if len(diffs_nao_nulas) > 0:
        _, p_valor = stats.wilcoxon(diffs_nao_nulas)
    else:
        p_valor = np.nan

    media_fmt = fmt_num_br(serie.mean())
    mediana_fmt = fmt_num_br(serie.median(), casas=0)
    p_fmt = fmt_p_br(p_valor)

    s = cod.lower()  # q11, q12, q13
    globals()[f"{s}_media"] = media_fmt
    globals()[f"{s}_med"] = mediana_fmt
    globals()[f"{s}_p"] = p_fmt

    resultados_bloco4.append(
        {
            "Item": f"`{cod}`",
            "Descrição": DESCRICOES.get(cod, cod),
            "_media": serie.mean(),
            "Média": media_fmt,
            "Mediana": mediana_fmt,
            "p vs. neutro": p_fmt,
        }
    )

n_bloco4 = max(ns_bloco4) if ns_bloco4 else len(pos)

# Observação: a consistência interna (Cronbach) de Q11+Q12 e a correlação de
# Spearman entre o escore conjunto e Q13 são calculadas uma única vez, na
# célula seguinte (alpha_q11_q12 / rho_q13), e não repetidas aqui.

df_bloco4_tbl = pd.DataFrame(resultados_bloco4)[
    ["Item", "Descrição", "Média", "Mediana", "p vs. neutro"]
]
Markdown(
    df_bloco4_tbl.to_markdown(
        index=False, colalign=("left", "left", "center", "center", "center")
    )
)

| Item   | Descrição                                           |  Média  |  Mediana  |  p vs. neutro  |
|:-------|:----------------------------------------------------|:-------:|:---------:|:--------------:|
| `Q11`  | Pensamento Crítico e Alucinações                    |  3,450  |     4     |     0,001      |
| `Q12`  | Transversalidade Pedagógica                         |  3,487  |     4     |     0,003      |
| `Q13`  | GLOBAL_BLOCO4 - Avaliação Geral da Integração de IA |  3,625  |     4     |     <0,001     |

Como apresentado na @tbl-ia-transversal, os três itens referentes à integração da inteligência artificial foram avaliados **significativamente acima do ponto neutro (3,0)**:

* **Pensamento Crítico e Alucinações (`Q11`)**: média `{python} q11_media`; mediana `{python} q11_med`; *p* = `{python} q11_p`;
* **Transversalidade Pedagógica (`Q12`)**: média `{python} q12_media`; mediana `{python} q12_med`; *p* = `{python} q12_p`;
* **Avaliação Geral da Integração de IA (`Q13`)**: média `{python} q13_media`; mediana `{python} q13_med`; *p* = `{python} q13_p`.


Os indicadores de consistência interna, associação entre as dimensões de
integração da IA e avaliação global, e comparação entre turmas foram
calculados para complementar a análise descritiva de `Q11`–`Q13`. Os
resultados são apresentados a seguir.

In [22]:
#| echo: false
#| output: true

import pandas as pd
from scipy import stats

# --- Cronbach alpha (Q11 + Q12) ---
dados_q11_q12 = pos[["Q11", "Q12"]].apply(pd.to_numeric, errors="coerce").dropna()
alpha_q11_q12 = cronbach_alpha(dados_q11_q12)
n_alpha_q11_q12 = len(dados_q11_q12)
alpha_q11_q12_fmt = fmt_num_br(alpha_q11_q12)

# --- Correlação de Spearman: escore médio (Q11+Q12) x Q13 ---
escore_q11_q12 = pos[["Q11", "Q12"]].apply(pd.to_numeric, errors="coerce").mean(axis=1)
dados_corr_q13 = pd.DataFrame(
    {
        "escore_q11_q12": escore_q11_q12,
        "q13": pd.to_numeric(pos["Q13"], errors="coerce"),
    }
).dropna()
n_corr_q13 = len(dados_corr_q13)
rho_q13, p_rho_q13 = stats.spearmanr(
    dados_corr_q13["escore_q11_q12"], dados_corr_q13["q13"]
)
rho_q13_fmt = fmt_num_br(rho_q13)
p_rho_q13_fmt = fmt_p_br(p_rho_q13)

# --- Kruskal-Wallis: Q13 por Turma ---
grupos_turma_q13 = [
    pd.to_numeric(grupo["Q13"], errors="coerce").dropna()
    for _, grupo in pos.groupby("Turma")
]
h_turma_q13, p_turma_q13 = stats.kruskal(*grupos_turma_q13)
h_q13_fmt = fmt_num_br(h_turma_q13)
p_q13_turma_fmt = fmt_p_br(p_turma_q13)

print(f"Cronbach (Q11+Q12, N={n_alpha_q11_q12}): alpha = {alpha_q11_q12_fmt}")
print(
    f"Spearman (escore Q11+Q12 x Q13, N={n_corr_q13}): rho = {rho_q13_fmt}, p = {p_rho_q13_fmt}"
)
print(f"Kruskal-Wallis Q13 por Turma: H = {h_q13_fmt}, p = {p_q13_turma_fmt}")

Cronbach (Q11+Q12, N=80): alpha = 0,786
Spearman (escore Q11+Q12 x Q13, N=80): rho = 0,777, p = <0,001
Kruskal-Wallis Q13 por Turma: H = 3,380, p = 0,184


A consistência interna entre `Q11` e `Q12` resultou em $\alpha$ de *Cronbach* [@cronbach1951] de `{python} alpha_q11_q12_fmt` ($n =$ `{python} n_alpha_q11_q12`). O escore conjunto dessas dimensões apresentou correlação de *Spearman* [@spearman1904] **forte e significativa** com a avaliação geral de `Q13` ($\rho =$ `{python} rho_q13_fmt`; *p* = `{python} p_rho_q13_fmt`), indicando que percepções mais favoráveis sobre o uso crítico e transversal da IA acompanham avaliações mais positivas de sua integração global.

Para `Q13`, o teste de *Kruskal--Wallis* [@kruskal1952] não identificou diferença estatisticamente significativa entre as turmas (*H* = `{python} h_q13_fmt`; *p* = `{python} p_q13_turma_fmt`). Assim, não há evidência de diferenças na avaliação global da integração da IA entre os diferentes períodos e níveis acadêmicos.


## Síntese dos Resultados

A @fig-boxplot-sintese-likert apresenta a distribuição das respostas dos **21 itens *Likert*** do questionário PÓS, ordenados pela média decrescente. A visualização permite comparar conjuntamente a tendência central e a dispersão das avaliações em relação ao ponto neutro da escala.

In [23]:
#| label: fig-boxplot-sintese-likert
#| fig-cap: "Distribuição das respostas dos 21 itens *Likert* do questionário PÓS, ordenados pela média decrescente. O Material Didático por IA, gerado com o *Gemini Notebook*, apresentou a pior média entre os itens analisados."
#| echo: false
#| output: true

import re
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

CODIGOS_SINTESE = (
    ["Q01", "Q02", "Q03", "Q04", "Q05", "Q06", "Q07"]
    + [f"Q08_{i:02d}" for i in range(1, 11)]
    + ["Q10", "Q11", "Q12", "Q13"]
)
assert (
    len(CODIGOS_SINTESE) == 21
), f"Esperava 21 itens Likert na síntese, obtive {len(CODIGOS_SINTESE)}."


def limpar_descricao_grafico(texto):
    """Remove colchetes, emojis e espaços duplicados de um rótulo, para uso em eixo de gráfico."""
    texto = str(texto)
    texto = texto.replace("[", "").replace("]", "")
    texto = re.sub(r"[\U00010000-\U0010FFFF]", "", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


# Monta a síntese
linhas_sintese = []
for cod in CODIGOS_SINTESE:
    serie = pd.to_numeric(pos[cod], errors="coerce").dropna()
    diffs = serie - VALOR_NEUTRO
    diffs_nao_nulas = diffs[diffs != 0]
    _, p_valor = (
        stats.wilcoxon(diffs_nao_nulas)
        if len(diffs_nao_nulas) > 0
        else (np.nan, np.nan)
    )
    linhas_sintese.append(
        {
            "item": cod,
            "descricao": DESCRICOES.get(cod, cod),
            "media": serie.mean(),
            "p_vs_neutro": p_valor,
        }
    )

df_sintese_geral = (
    pd.DataFrame(linhas_sintese)
    .sort_values("media", ascending=False)
    .reset_index(drop=True)
)
df_sintese_geral["descricao_grafico"] = df_sintese_geral["descricao"].apply(
    limpar_descricao_grafico
)

# Variáveis escalares dinâmicas para o texto
item_pior = df_sintese_geral.iloc[-1]
item_pior_codigo = item_pior["item"]
item_pior_nome = item_pior["descricao"]
item_pior_media_fmt = fmt_num_br(item_pior["media"])
item_pior_p_fmt = fmt_p_br(item_pior["p_vs_neutro"])

itens_sem_evidencia = df_sintese_geral[df_sintese_geral["p_vs_neutro"] >= 0.05]
qtd_itens_sintese = len(df_sintese_geral)

print(
    f"Pior média: {item_pior_codigo} ({item_pior_nome}) = {item_pior_media_fmt}"
)
if len(itens_sem_evidencia) > 0:
    print(
        "Itens sem evidência de diferença ao ponto neutro:",
        ", ".join(itens_sem_evidencia["item"]),
    )

# ------------------------------------------------------------------------------
# Dados para o boxplot
# ------------------------------------------------------------------------------
dados_boxplot = []
for _, row in df_sintese_geral.iterrows():
    respostas = pd.to_numeric(pos[row["item"]], errors="coerce").dropna()
    dados_boxplot.append(
        pd.DataFrame(
            {
                "item": row["descricao_grafico"],
                "codigo": row["item"],
                "resposta": respostas,
            }
        )
    )
dados_boxplot = pd.concat(dados_boxplot, ignore_index=True)

ordem_descricoes = df_sintese_geral["descricao_grafico"].tolist()
dados_boxplot["item"] = pd.Categorical(
    dados_boxplot["item"], categories=ordem_descricoes, ordered=True
)

paleta_pastel = [
    "#AEC6CF",
    "#BFD8B8",
    "#F7C6A3",
    "#CDB4DB",
    "#FFD6A5",
    "#BDE0FE",
    "#CDEAC0",
    "#F6BDC0",
    "#D8BFD8",
    "#B8E0D2",
    "#F9D5A7",
    "#C6DEF1",
    "#E2C2FF",
    "#C7CEEA",
    "#FFDAC1",
    "#B5EAD7",
    "#FFB7B2",
    "#D4A5A5",
    "#AFCBFF",
    "#C9E4DE",
    "#E8D5B7",
]
cores_itens = dict(zip(ordem_descricoes, paleta_pastel))

fig, ax = plt.subplots(figsize=(11.5, 8.5))
sns.boxplot(
    data=dados_boxplot,
    y="item",
    x="resposta",
    order=ordem_descricoes,
    hue="item",
    palette=cores_itens,
    legend=False,
    showmeans=True,
    meanprops={
        "marker": "D",
        "markerfacecolor": "white",
        "markeredgecolor": "black",
        "markersize": 5,
        "zorder": 10,
    },
    width=0.58,
    linewidth=1.0,
    fliersize=2.5,
    ax=ax,
)
ax.axvline(
    x=VALOR_NEUTRO,
    color="#555555",
    linestyle="--",
    linewidth=1.1,
    alpha=0.85,
    zorder=1,
)

for i, (_, row) in enumerate(df_sintese_geral.iterrows()):
    ax.text(
        row["media"] + 0.06,
        i,
        f"{row['media']:.2f}".replace(".", ","),
        color=CORES["texto_medio"],
        fontsize=8.5,
        fontweight="bold",
        va="center",
        ha="left",
        zorder=12,
    )

ax.set_xlim(0.5, 5.6)
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_xlabel(
    "Resposta Likert (1–5)",
    fontsize=FONTE["eixo"],
    fontweight="bold",
    labelpad=8,
)
ax.set_ylabel("")
ax.set_title(
    "Distribuição dos Itens Likert — Avaliação PÓS",
    fontsize=FONTE["titulo"],
    fontweight="bold",
    pad=10,
)

marcador_media = mlines.Line2D(
    [],
    [],
    color="white",
    marker="D",
    markerfacecolor="white",
    markeredgecolor="black",
    markersize=5,
    label="Média",
)
linha_neutra_legenda = mlines.Line2D(
    [],
    [],
    color="#555555",
    linestyle="--",
    linewidth=1.1,
    label="Ponto neutro = 3,0",
)
ax.legend(
    handles=[marcador_media, linha_neutra_legenda],
    loc="lower right",
    frameon=True,
    facecolor="white",
    framealpha=0.9,
    fontsize=FONTE["legenda"],
)

ax.grid(axis="x", linestyle="--", linewidth=0.6, alpha=0.30)
ax.grid(axis="y", visible=False)
ax.tick_params(axis="y", labelsize=8.5)
ax.tick_params(axis="x", labelsize=9)
estilo_eixos(ax)

plt.tight_layout()
plt.show()
plt.close(fig)

Pior média: Q08_02 (Material Didático por IA) = 3,087
Itens sem evidência de diferença ao ponto neutro: Q08_02


<Figure size 3450x2550 with 1 Axes>

A síntese visual dos `{python} qtd_itens_sintese` itens *Likert* (@fig-boxplot-sintese-likert) reúne os principais padrões observados na avaliação final:

1. **Predomínio da prática aplicada:** atividades de resolução orientada, programação e simulações interativas concentram as avaliações mais elevadas e as maiores frequências de seleção.
2. **Estabilidade na análise pareada:** nenhuma das dimensões comparadas entre PRÉ e PÓS apresentou diferença estatisticamente significativa no teste pareado ($p > 0,05$).
3. **Avaliação contextual da IA:** a integração transversal da IA foi avaliada positivamente, enquanto os recursos com participação direta na geração de conteúdo, *feedback* ou rubricas apresentaram avaliações mais moderadas.

Entre os itens analisados, **`{python} item_pior_nome` (`{python} item_pior_codigo`)** apresentou a menor média (`{python} item_pior_media_fmt`) e foi o único sem diferença estatisticamente significativa em relação ao ponto neutro (*p* = `{python} item_pior_p_fmt`). Esse resultado descreve o padrão observado nas avaliações, mas **não permite atribuir causalidade ao uso da IA**, uma vez que os recursos diferem em finalidade, formato e contexto de utilização.


### Tabela Síntese dos Resultados Estatísticos {.unnumbered}

Como síntese quantitativa, a @tbl-sintese-geral reúne os **21 itens *Likert*** dos Blocos 2, 3 e 4 do questionário PÓS, ordenados pela média decrescente. São apresentados, para cada item, a média, o desvio-padrão (DP), a mediana e o *p*-valor do teste de postos sinalizados de *Wilcoxon* para uma amostra, tendo **3,0 como ponto neutro** da escala. O nível de significância adotado foi de 5%.

In [24]:
#| label: tbl-sintese-geral
#| tbl-cap: "Síntese estatística dos itens *Likert* dos Blocos 2, 3 e 4, ordenada pela média decrescente. Os valores de p referem-se ao teste de postos sinalizados de Wilcoxon para uma amostra, tendo 3,0 como valor de referência."
#| echo: false
#| output: true

import re
import numpy as np
import pandas as pd
from IPython.display import Markdown
from scipy import stats


def classificar_bloco(cod):
    rotulo_bruto = METADADOS_PERGUNTAS.get(cod, {}).get("rotulo", "") or ""

    m = re.search(r"GLOBAL_BLOCO(\d)", rotulo_bruto)
    if m:
        destino = {
            "2": "Competências",
            "3": "Recursos Didáticos",
            "4": "IA Transversal",
        }
        return f"Global — {destino.get(m.group(1), '?')}"

    if re.fullmatch(r"Q0[1-6]", cod):
        return "Competências"
    if re.fullmatch(r"Q08_\d{2}", cod):
        return "Recursos Didáticos"
    if cod in ("Q11", "Q12"):
        return "IA Transversal"
    return "Outro"


codigos_sintese = (
    ["Q01", "Q02", "Q03", "Q04", "Q05", "Q06", "Q07"]
    + [f"Q08_{i:02d}" for i in range(1, 11)]
    + ["Q10", "Q11", "Q12", "Q13"]
)
assert len(codigos_sintese) == 21

blocos_detectados = {cod: classificar_bloco(cod) for cod in codigos_sintese}
assert "Outro" not in blocos_detectados.values(), (
    f"Item(ns) sem bloco reconhecido: {[c for c, b in blocos_detectados.items() if b == 'Outro']}"
)

linhas_sintese_geral = []
ns_sintese = set()
for cod in codigos_sintese:
    serie = pd.to_numeric(pos[cod], errors="coerce").dropna()
    ns_sintese.add(len(serie))
    diffs = serie - VALOR_NEUTRO
    diffs_nao_nulas = diffs[diffs != 0]
    _, p_valor = (
        stats.wilcoxon(diffs_nao_nulas)
        if len(diffs_nao_nulas) > 0
        else (np.nan, np.nan)
    )

    linhas_sintese_geral.append(
        {
            "Bloco": blocos_detectados[cod],
            "Item": cod,
            "Descrição resumida": DESCRICOES.get(cod, cod),
            "_media": serie.mean(),
            "DP": fmt_num_br(serie.std()),
            "Mediana": fmt_num_br(serie.median(), casas=0),
            "_p": p_valor,
        }
    )

if len(ns_sintese) > 1:
    print(f"Aviso: N variável entre os 21 itens da síntese -> {sorted(ns_sintese)}")
n_sintese = max(ns_sintese) if ns_sintese else len(pos)

df_sintese_completa = (
    pd.DataFrame(linhas_sintese_geral)
    .sort_values("_media", ascending=False)
    .reset_index(drop=True)
)

# Estatísticas gerais para interpolação
n_significativos = int((df_sintese_completa["_p"] < 0.05).sum())
n_total_sintese = len(df_sintese_completa)
pct_significativos_fmt = (
    f"{n_significativos / n_total_sintese:.1%}".replace(".", ",")
)

# Identificação da exceção não significativa
item_excecao = df_sintese_completa[df_sintese_completa["_p"] >= 0.05].iloc[0]
excecao_cod = item_excecao["Item"]
excecao_nome = item_excecao["Descrição resumida"]
excecao_media = fmt_num_br(item_excecao["_media"])
excecao_p = fmt_p_br(item_excecao["_p"])

df_sintese_completa["Item"] = df_sintese_completa["Item"].apply(
    lambda c: f"`{c}`"
)
df_sintese_completa["Média"] = df_sintese_completa["_media"].apply(fmt_num_br)
df_sintese_completa["p vs. 3,0"] = df_sintese_completa["_p"].apply(fmt_p_br)
df_sintese_completa["Sig.?"] = df_sintese_completa["_p"].apply(
    lambda p: "Sim" if p < 0.05 else "Não"
)

df_sintese_tbl = df_sintese_completa[
    [
        "Bloco",
        "Item",
        "Descrição resumida",
        "Média",
        "DP",
        "Mediana",
        "p vs. 3,0",
        "Sig.?",
    ]
]

print(
    f"{n_significativos} de {n_total_sintese} itens ({pct_significativos_fmt}) com evidência de deslocamento favorável (N={n_sintese})."
)

Markdown(
    df_sintese_tbl.to_markdown(
        index=False,
        colalign=(
            "left",
            "left",
            "left",
            "center",
            "center",
            "center",
            "center",
            "center",
        ),
    )
)

20 de 21 itens (95,2%) com evidência de deslocamento favorável (N=80).


| Bloco                       | Item     | Descrição resumida                                     |  Média  |  DP   |  Mediana  |  p vs. 3,0  |  Sig.?  |
|:----------------------------|:---------|:-------------------------------------------------------|:-------:|:-----:|:---------:|:-----------:|:-------:|
| Recursos Didáticos          | `Q08_04` | Resolução de EPs em Aula                               |  4,588  | 0,650 |     5     |   <0,001    |   Sim   |
| Competências                | `Q01`    | PÓS - Capacidade de Programação                        |  4,562  | 0,760 |     5     |   <0,001    |   Sim   |
| Recursos Didáticos          | `Q08_03` | Simuladores Interativos                                |  4,362  | 0,846 |     5     |   <0,001    |   Sim   |
| Recursos Didáticos          | `Q08_05` | Biblioteca morph.py                                    |  4,300  | 1,048 |     5     |   <0,001    |   Sim   |
| Recursos Didáticos          | `Q08_01` | Livro Interativo                                       |  4,287  | 0,814 |     4     |   <0,001    |   Sim   |
| Recursos Didáticos          | `Q08_10` | Uso do TestSuite.py                                    |  4,263  | 0,938 |     5     |   <0,001    |   Sim   |
| Recursos Didáticos          | `Q08_09` | Correção Objetiva VPL                                  |  4,213  | 1,015 |     5     |   <0,001    |   Sim   |
| Global — Competências       | `Q07`    | GLOBAL_BLOCO2 - Avaliação Geral do Aprendizado         |  4,125  | 0,891 |     4     |   <0,001    |   Sim   |
| Recursos Didáticos          | `Q08_06` | Provas e Simulados Paramétricos                        |  4,075  | 1,077 |     4     |   <0,001    |   Sim   |
| Global — Recursos Didáticos | `Q10`    | GLOBAL_BLOCO3 - Avaliação Geral dos Recursos Didáticos |  4,075  | 0,823 |     4     |   <0,001    |   Sim   |
| Competências                | `Q05`    | PÓS - Autonomia com Bibliotecas                        |  3,825  | 0,925 |     4     |   <0,001    |   Sim   |
| Competências                | `Q03`    | PÓS - Atuação Profissional                             |  3,725  | 1,043 |     4     |   <0,001    |   Sim   |
| Competências                | `Q02`    | PÓS - Atuação Acadêmica                                |  3,712  | 1,009 |     4     |   <0,001    |   Sim   |
| Global — IA Transversal     | `Q13`    | GLOBAL_BLOCO4 - Avaliação Geral da Integração de IA    |  3,625  | 1,236 |     4     |   <0,001    |   Sim   |
| Recursos Didáticos          | `Q08_08` | Rubrica por IA em Simulados/Provas                     |  3,587  | 1,099 |     4     |   <0,001    |   Sim   |
| Competências                | `Q04`    | PÓS - Fundamentos Matemáticos                          |  3,538  | 1,043 |     4     |   <0,001    |   Sim   |
| IA Transversal              | `Q12`    | Transversalidade Pedagógica                            |  3,487  | 1,212 |     4     |    0,003    |   Sim   |
| IA Transversal              | `Q11`    | Pensamento Crítico e Alucinações                       |  3,450  | 1,135 |     4     |    0,001    |   Sim   |
| Competências                | `Q06`    | PÓS - Feedback por IA                                  |  3,438  | 1,330 |     4     |    0,007    |   Sim   |
| Recursos Didáticos          | `Q08_07` | Feedback Socrático no VPL                              |  3,388  | 1,248 |     4     |    0,011    |   Sim   |
| Recursos Didáticos          | `Q08_02` | Material Didático por IA                               |  3,087  | 1,361 |     3     |    0,710    |   Não   |

Considerando a amostra do PÓS ($N =$ `{python} n_sintese`), **`{python} n_significativos` dos `{python} n_total_sintese` itens (`{python} pct_significativos_fmt`) apresentaram avaliação estatisticamente superior ao ponto neutro** (@tbl-sintese-geral). A única exceção foi **`{python} excecao_nome` (`{python} excecao_cod`)**, com média de `{python} excecao_media` e sem diferença significativa em relação a 3,0 (*p* = `{python} excecao_p`).

Em conjunto, as médias mais elevadas concentram-se em **atividades práticas, programação e recursos interativos**, enquanto os recursos baseados diretamente em IA apresentaram avaliações mais moderadas. Esse panorama integra, em uma mesma síntese, as percepções sobre competências, recursos didáticos e integração da IA na disciplina.


## Principais Conclusões

Os resultados sintetizam os principais achados da avaliação da disciplina:

1. **Avaliação global positiva:** `Q10` apresentou média `{python} media_q10_fmt` e mediana `{python} mediana_q10_fmt`.

2. **Diferenciação entre os recursos:** o teste de Friedman indicou diferenças significativas entre os dez recursos ($\chi^2$(`{python} df_friedman`) = `{python} chi2_friedman_fmt`; *p* = `{python} p_friedman_fmt`), com coeficiente de concordância de Kendall $W =$ `{python} w_kendall_fmt`.

3. **Valorização da prática aplicada:** **`{python} pref1_nome`**, **`{python} pref2_nome`** e **`{python} pref3_nome`** estiveram entre os recursos mais bem avaliados e mais selecionados em `Q09`, destacando a importância da prática, da programação e da interatividade.

4. **Estabilidade na comparação PRÉ--PÓS:** entre os `{python} n_pareados` estudantes identificados nos dois momentos, o teste de *Wilcoxon* não indicou mudanças estatisticamente significativas nas dimensões analisadas ($p > 0,05$).

5. **Integração da IA avaliada favoravelmente:** `Q11`--`Q13` apresentaram avaliações superiores ao ponto neutro, embora os recursos com participação direta da IA na geração de conteúdo, *feedback* ou rubricas tenham recebido avaliações mais moderadas.

6. **Associação entre avaliações específicas e globais:** o escore médio de `Q08` apresentou correlação moderada com `Q10` ($\rho =$ `{python} rho_q08_q10_fmt`; *p* = `{python} p_rho_q08_q10_fmt`), enquanto as dimensões `Q11`--`Q12` apresentaram associação forte com `Q13` ($\rho =$ `{python} rho_q13_fmt`; *p* = `{python} p_rho_q13_fmt`).

7. **Ausência de diferenças significativas entre turmas:** os testes de *Kruskal--Wallis* não indicaram diferenças entre os grupos para `Q10` (*p* = `{python} p_q10_turma_fmt`) ou `Q13` (*p* = `{python} p_q13_turma_fmt`).

8. **Consistência interna adequada:** os itens de `Q08` apresentaram $\alpha =$ `{python} alpha_q08_fmt`, enquanto `Q11`--`Q12` apresentaram $\alpha =$ `{python} alpha_q11_q12_fmt`.

Em conjunto, os resultados destacam a **prática aplicada, a programação e a interatividade** como elementos centrais da experiência pedagógica. A IA apresentou avaliação favorável quando integrada de forma transversal ao processo de aprendizagem, mas seus diferentes usos receberam avaliações distintas, o que reforça a importância do **contexto e da finalidade pedagógica** de cada recurso.

::: {.callout-note}

## Nota metodológica {.unnumbered}

Os resultados descrevem **percepções, diferenças e associações estatísticas**, não estabelecendo relações causais. A análise longitudinal está restrita aos `{python} n_pareados` estudantes da amostra pareada, enquanto as comparações entre as amostras completas PRÉ e PÓS refletem diferenças entre os respectivos grupos de respondentes.

:::


## Avaliação de Valência das Respostas Abertas — Q14 e Q15

As questões abertas `Q14` e `Q15` complementam as avaliações estruturadas do questionário PÓS, permitindo analisar espontaneamente aspectos da experiência pedagógica. A **análise temática** identifica *quais* aspectos foram mencionados, enquanto a **análise de valência** caracteriza sua orientação avaliativa.

Como as questões possuem finalidades distintas — `Q14` focaliza **ganhos percebidos de aprendizagem**, enquanto `Q15` aborda **dificuldades e possibilidades de aprimoramento** —, seus resultados são analisados separadamente e não constituem medidas diretamente comparáveis de satisfação geral.

### Unidade de análise e critérios

A unidade de análise foi a **resposta completa**, classificada como **positiva, negativa, mista ou neutra** conforme a orientação predominante. Os critérios adotados para cada categoria são apresentados na @tbl-criterios-valencia. Quando uma resposta apresentava avaliações distintas sobre diferentes aspectos, realizou-se também o refinamento por tema.

In [25]:
#| label: tbl-criterios-valencia
#| tbl-cap: "Critérios de classificação da valência das respostas abertas (Q14/Q15)."
#| echo: false

df_criterios_valencia = pd.DataFrame([
    {"Código": "**POSITIVA**", "Critério": "Predominam benefício, aprovação ou melhoria percebida."},
    {"Código": "**NEGATIVA**", "Critério": "Predominam crítica, dificuldade ou ausência de benefício."},
    {"Código": "**MISTA**", "Critério": "Aspectos positivos e negativos relevantes coexistem explicitamente."},
    {"Código": "**NEUTRA**", "Critério": "A resposta apenas descreve ou identifica um elemento, sem avaliação suficiente para estabelecer polaridade."},
])

Markdown(df_criterios_valencia.to_markdown(index=False, colalign=("left", "left")))

| Código       | Critério                                                                                                    |
|:-------------|:------------------------------------------------------------------------------------------------------------|
| **POSITIVA** | Predominam benefício, aprovação ou melhoria percebida.                                                      |
| **NEGATIVA** | Predominam crítica, dificuldade ou ausência de benefício.                                                   |
| **MISTA**    | Aspectos positivos e negativos relevantes coexistem explicitamente.                                         |
| **NEUTRA**   | A resposta apenas descreve ou identifica um elemento, sem avaliação suficiente para estabelecer polaridade. |

A @tbl-resumo-q14-q15 apresenta a síntese da análise das respostas abertas. Foram consideradas apenas as **respostas não vazias**, enquanto a codificação temática permitiu múltiplas menções por resposta. Assim, os quantitativos de respostas e de menções temáticas são apresentados separadamente, evitando sua interpretação como medidas equivalentes.

In [26]:
#| echo: false
#| output: false

from collections import Counter

# ==========================================================================
# CODIFICAÇÃO TEMÁTICA MANUAL — Q14 e Q15
# --------------------------------------------------------------------------
# Codificação múltipla (uma resposta pode receber mais de um tema), feita
# por leitura integral de cada resposta não vazia. Não é extraível
# automaticamente dos dados: pos["Q14"]/pos["Q15"] vêm 100% NaN no CSV
# anonimizado — o texto bruto das respostas abertas foi removido na
# anonimização, por conter potencial informação identificável. Os índices
# abaixo correspondem à posição de cada resposta não vazia, na ordem em que
# apareciam no CSV original antes da anonimização.
# ==========================================================================

q14_codes = {
    0: ["A"], 1: ["A"], 2: ["A"], 3: ["B"], 4: ["A"], 5: ["C"], 6: ["A"], 7: ["A"],
    8: ["A"], 9: ["A"], 10: ["A"], 11: ["C"], 12: ["A"], 13: ["C"], 14: ["A"], 15: ["A"],
    16: ["B"], 17: ["B"], 18: ["A"], 19: ["A", "D"], 20: ["A"], 21: ["D", "C"], 22: ["B"],
    23: ["D"], 24: ["A"], 25: ["B"], 26: ["A"], 27: ["A"], 28: ["A"], 29: ["A"], 30: ["E"],
    31: ["A"], 32: ["D", "B"], 33: ["F"], 34: ["D"], 35: ["E"], 36: ["A"], 37: ["F"],
    38: ["D"], 39: ["D", "A"],
}
labels_q14 = {
    "A": "Geração de material (livro/PDF/vídeos/simuladores/NotebookLM)",
    "B": "Feedback de IA nos EPs",
    "C": "Feedback/rubrica em simulados e provas",
    "D": "EPs/prática em geral (sem menção clara à IA)",
    "E": "Nenhum ganho / rejeita uso de IA",
    "F": "Não classificável / resposta não conclusiva",
}

q15_codes = {
    0: ["SEB", "TEMPO"], 1: ["MCTEST", "VPL"], 2: ["VPL", "MATERIAL"], 3: ["FEEDBACK"],
    4: ["SEB"], 5: ["BIBLIOTECA"], 6: ["VPL"], 7: ["FEEDBACK"], 8: ["VPL"],
    9: ["BIBLIOTECA", "VPL"], 10: ["FEEDBACK"], 11: ["BIBLIOTECA", "FEEDBACK"],
    12: ["SEB", "TEMPO", "BIBLIOTECA"], 13: ["MCTEST"], 14: ["FEEDBACK", "VPL"],
    15: ["SEB", "TEMPO", "BIBLIOTECA"], 16: ["SEB"], 17: ["VPL"], 18: ["SEB", "VPL"],
    19: ["BIBLIOTECA"], 20: ["VPL"], 21: ["FEEDBACK"], 22: ["MATERIAL"], 23: ["NENHUM"],
    24: ["FEEDBACK", "MATERIAL"], 25: ["FEEDBACK", "VPL"], 26: ["OUTRO"],
    27: ["TEMPO", "OUTRO"], 28: ["FEEDBACK"], 29: ["MATERIAL", "SEB"],
    30: ["BIBLIOTECA", "OUTRO"], 31: ["VPL"], 32: ["MCTEST"], 33: ["FEEDBACK", "BIBLIOTECA"],
    34: ["FEEDBACK", "BIBLIOTECA"], 35: ["SEB", "TEMPO"], 36: ["FEEDBACK", "MATERIAL"],
    37: ["OUTRO", "MATERIAL"],
}
labels_q15 = {
    "SEB": "SEB (ambiente seguro de provas)",
    "MCTEST": "MCTest",
    "TEMPO": "Tempo de prova/simulado",
    "FEEDBACK": "Feedback por IA",
    "BIBLIOTECA": "Biblioteca (morph/opencv)",
    "VPL": "VPL / restrição à rede da UFABC (emergente)",
    "MATERIAL": "Material gerado por IA (vídeos/slides/excesso) (emergente)",
    "NENHUM": "Nenhuma dificuldade relevante",
    "OUTRO": "Outro (estrutura dos EPs, conteúdo, etc.)",
}

print(f"q14_codes: {len(q14_codes)} respostas codificadas | q15_codes: {len(q15_codes)} respostas codificadas.")


def tabela_tematica(codigos_dict, labels_dict, coluna_nome, prefixo_var):
    """Monta a tabela de distribuição temática (usada tanto para Q14 quanto
    para Q15) a partir do dicionário de codificação manual, e exporta as
    variáveis escalares (ex.: t14_1_nome, t14_1_n, t14_1_pct, ...) usadas
    inline no texto em Markdown.
    """
    n_respondentes = len(codigos_dict)
    assert n_respondentes > 0, (
        f"{prefixo_var}_codes vazio — confira se a codificação manual foi "
        "carregada antes deste chunk."
    )

    contagem = Counter()
    for codigos in codigos_dict.values():
        for c in codigos:
            contagem[c] += 1

    linhas = []
    for idx, (cat, n) in enumerate(contagem.most_common()):
        pos_idx = idx + 1
        nome_tema = labels_dict.get(cat, cat)
        pct_str = (
            f"{n / n_respondentes:.1%}".replace(".", ",")
            if n_respondentes > 0
            else "—"
        )

        globals()[f"{prefixo_var}_{pos_idx}_nome"] = nome_tema
        globals()[f"{prefixo_var}_{pos_idx}_n"] = n
        globals()[f"{prefixo_var}_{pos_idx}_pct"] = pct_str

        linhas.append(
            {
                coluna_nome: nome_tema,
                "Menções (n)": n,
                "Frequência Relativa*": pct_str,
            }
        )

    return pd.DataFrame(linhas), n_respondentes


q14_codes: 40 respostas codificadas | q15_codes: 38 respostas codificadas.


In [27]:
#| label: tbl-resumo-q14-q15
#| tbl-cap: "Síntese da unidade de análise: respostas não vazias e menções temáticas (codificação múltipla) em Q14 e Q15."
#| echo: false
#| output: true

import pandas as pd
from IPython.display import Markdown

N_TOTAL_QUESTIONARIO = len(pos)

linhas_resumo_q14_q15 = []
for cod, codes_dict in [("Q14", q14_codes), ("Q15", q15_codes)]:
    n_respostas = len(codes_dict)
    n_mencoes = sum(len(codigos) for codigos in codes_dict.values())
    pct_val = 100 * n_respostas / N_TOTAL_QUESTIONARIO
    pct_fmt = f"{pct_val:.1f}%".replace(".", ",")

    s = cod.lower()  # q14, q15
    globals()[f"{s}_n_resp"] = n_respostas
    globals()[f"{s}_pct_resp"] = pct_fmt
    globals()[f"{s}_n_mencoes"] = n_mencoes

    linhas_resumo_q14_q15.append(
        {
            "Item": f"`{cod}`",
            "N respostas não vazias": n_respostas,
            "% do total (N={})".format(N_TOTAL_QUESTIONARIO): pct_fmt,
            "Menções temáticas": n_mencoes,
        }
    )

df_resumo_q14_q15 = pd.DataFrame(linhas_resumo_q14_q15)

assert (df_resumo_q14_q15["N respostas não vazias"] > 0).all(), (
    "q14_codes/q15_codes vazios — confira se esses dicionários foram "
    "definidos antes deste chunk (bloco de codificação manual)."
)

Markdown(
    df_resumo_q14_q15.to_markdown(
        index=False, colalign=("left", "center", "center", "center")
    )
)

| Item   |  N respostas não vazias  |  % do total (N=80)  |  Menções temáticas  |
|:-------|:------------------------:|:-------------------:|:-------------------:|
| `Q14`  |            40            |        50,0%        |         44          |
| `Q15`  |            38            |        47,5%        |         59          |

Foram analisadas **`{python} q14_n_resp` respostas válidas em `Q14`** (`{python} q14_pct_resp` de $N = `{python} N_TOTAL_QUESTIONARIO`$) e **`{python} q15_n_resp` em `Q15`** (`{python} q15_pct_resp`), conforme apresentado na @tbl-resumo-q14-q15.

A **codificação temática múltipla** identificou **`{python} q14_n_mencoes` menções em `Q14`** e **`{python} q15_n_mencoes` em `Q15`**. Como uma resposta pode conter mais de um tema, esses quantitativos representam **ocorrências temáticas**, e não o número de respostas.


### Análise temática de Q14 — Maior contribuição da IA

A @tbl-analise-tematica-q14 apresenta a distribuição dos **temas identificados nas respostas à `Q14`**, que solicitou aos estudantes a indicação da principal contribuição da IA para sua aprendizagem. A codificação permitiu registrar mais de um tema na mesma resposta; por isso, as frequências representam **menções temáticas**, e não necessariamente participantes distintos.


In [28]:
#| label: tbl-analise-tematica-q14
#| tbl-cap: "Distribuição temática das percepções sobre a maior contribuição da IA (Q14)."
#| echo: false
#| output: true

from IPython.display import Markdown

df_q14_temas, n_respondentes_q14 = tabela_tematica(
    q14_codes, labels_q14, "Eixo Temático / Categoria", "t14"
)

Markdown(
    df_q14_temas.to_markdown(
        index=False, colalign=("left", "center", "center")
    )
)

| Eixo Temático / Categoria                                     |  Menções (n)  |  Frequência Relativa*  |
|:--------------------------------------------------------------|:-------------:|:----------------------:|
| Geração de material (livro/PDF/vídeos/simuladores/NotebookLM) |      23       |         57,5%          |
| EPs/prática em geral (sem menção clara à IA)                  |       7       |         17,5%          |
| Feedback de IA nos EPs                                        |       6       |         15,0%          |
| Feedback/rubrica em simulados e provas                        |       4       |         10,0%          |
| Nenhum ganho / rejeita uso de IA                              |       2       |          5,0%          |
| Não classificável / resposta não conclusiva                   |       2       |          5,0%          |

Como detalhado na @tbl-analise-tematica-q14, o eixo mais representativo foi **`{python} t14_1_nome`**, com `{python} t14_1_n` menções (`{python} t14_1_pct`). Em seguida, destacaram-se **`{python} t14_2_nome`** (`{python} t14_2_pct`), **`{python} t14_3_nome`** (`{python} t14_3_pct`) e **`{python} t14_4_nome`** (`{python} t14_4_pct`). Já **`{python} t14_5_nome`** correspondeu a `{python} t14_5_pct` das menções, enquanto `{python} t14_6_pct` foram classificadas como inconclusivas.

Em conjunto, os resultados indicam que a principal contribuição percebida da IA em `Q14` esteve relacionada à **produção e organização de materiais didáticos**, seguida por usos de apoio à aprendizagem e *feedback*.


### Principais dificuldades e sugestões de ajuste — Q15

A análise temática de `Q15` identificou as principais **dificuldades percebidas e sugestões de ajuste metodológico** apresentadas pelos estudantes. Como uma mesma resposta podia mencionar diferentes aspectos, foi utilizada codificação múltipla; portanto, as frequências correspondem a **menções temáticas**, e não ao número de respondentes. Os resultados são apresentados na @tbl-analise-tematica-q15.


In [29]:
#| label: tbl-analise-tematica-q15
#| tbl-cap: "Principais dificuldades e sugestões de ajustes metodológicos apontadas pelos estudantes (Q15)."
#| echo: false
#| output: true

from IPython.display import Markdown

df_q15_temas, n_respondentes_q15 = tabela_tematica(
    q15_codes, labels_q15, "Dificuldade / Sugestão de Ajuste", "t15"
)

Markdown(
    df_q15_temas.to_markdown(
        index=False, colalign=("left", "center", "center")
    )
)

| Dificuldade / Sugestão de Ajuste                           |  Menções (n)  |  Frequência Relativa*  |
|:-----------------------------------------------------------|:-------------:|:----------------------:|
| Feedback por IA                                            |      12       |         31,6%          |
| VPL / restrição à rede da UFABC (emergente)                |      11       |         28,9%          |
| Biblioteca (morph/opencv)                                  |       9       |         23,7%          |
| SEB (ambiente seguro de provas)                            |       8       |         21,1%          |
| Material gerado por IA (vídeos/slides/excesso) (emergente) |       6       |         15,8%          |
| Tempo de prova/simulado                                    |       5       |         13,2%          |
| Outro (estrutura dos EPs, conteúdo, etc.)                  |       4       |         10,5%          |
| MCTest                                                     |       3       |          7,9%          |
| Nenhuma dificuldade relevante                              |       1       |          2,6%          |

Conforme consolidado na @tbl-analise-tematica-q15, as dificuldades e sugestões de ajuste concentraram-se principalmente em **`{python} t15_1_nome`** (`{python} t15_1_n` menções; `{python} t15_1_pct`), **`{python} t15_2_nome`** (`{python} t15_2_n`; `{python} t15_2_pct`), **`{python} t15_3_nome`** (`{python} t15_3_n`; `{python} t15_3_pct`) e **`{python} t15_4_nome`** (`{python} t15_4_n`; `{python} t15_4_pct`). Também foram registradas ocorrências relacionadas a **`{python} t15_5_nome`** (`{python} t15_5_n`; `{python} t15_5_pct`), **`{python} t15_6_nome`** (`{python} t15_6_n`; `{python} t15_6_pct`) e outros aspectos estruturais.

Em conjunto, os apontamentos indicam **oportunidades concretas de aprimoramento**, especialmente quanto ao *feedback* por IA, à infraestrutura de submissão, às bibliotecas utilizadas, ao ambiente de avaliação e à adequação do tempo disponível.


### Síntese da Análise Temática de `Q14` e `Q15`

A distribuição comparativa dos principais eixos temáticos identificados nas duas questões é apresentada na @fig-sintese-qualitativa-q14-q15. A visualização permite contrastar, de forma conjunta, os aspectos associados aos **ganhos percebidos** (`Q14`) e às **dificuldades ou sugestões de ajuste** (`Q15`).


In [30]:
#| label: fig-sintese-qualitativa-q14-q15
#| fig-cap: "Distribuição comparativa dos principais eixos temáticos identificados nas questões abertas Q14 e Q15."
#| echo: false
#| output: true

import textwrap
import matplotlib.pyplot as plt

assert (
    not df_q14_temas.empty
), "df_q14_temas vazio — rode o chunk tbl-analise-tematica-q14 antes deste."
assert (
    not df_q15_temas.empty
), "df_q15_temas vazio — rode o chunk tbl-analise-tematica-q15 antes deste."

LARGURA_ROTULO = 38  # caracteres por linha antes de quebrar


def quebrar_rotulos(serie_categorias, largura=LARGURA_ROTULO):
    return [textwrap.fill(str(c), width=largura) for c in serie_categorias]


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.5), sharey=False)

# Gráfico Q14
df_plot_q14 = df_q14_temas[df_q14_temas["Menções (n)"] > 0].sort_values(
    "Menções (n)", ascending=True
)
y1 = quebrar_rotulos(df_plot_q14["Eixo Temático / Categoria"])
bars1 = ax1.barh(
    y1,
    df_plot_q14["Menções (n)"],
    color=CORES["aprovado"],
    height=0.6,
    zorder=3,
)
for bar, n in zip(bars1, df_plot_q14["Menções (n)"]):
    ax1.text(
        bar.get_width() + 0.15,
        bar.get_y() + bar.get_height() / 2,
        str(n),
        va="center",
        ha="left",
        fontsize=FONTE["texto"],
        color=CORES["texto_escuro"],
    )
ax1.set_title(
    "Q14: Ganhos com a IA", fontsize=FONTE["subtitulo"], weight="bold"
)
ax1.set_xlabel("Número de Menções", fontsize=FONTE["eixo"])
ax1.set_xlim(0, df_plot_q14["Menções (n)"].max() * 1.18)
ax1.tick_params(axis="y", labelsize=FONTE["texto"])
estilo_eixos(ax1)

# Gráfico Q15
df_plot_q15 = df_q15_temas[df_q15_temas["Menções (n)"] > 0].sort_values(
    "Menções (n)", ascending=True
)
y2 = quebrar_rotulos(df_plot_q15["Dificuldade / Sugestão de Ajuste"])
bars2 = ax2.barh(
    y2, df_plot_q15["Menções (n)"], color=CORES["outlier"], height=0.6, zorder=3
)
for bar, n in zip(bars2, df_plot_q15["Menções (n)"]):
    ax2.text(
        bar.get_width() + 0.15,
        bar.get_y() + bar.get_height() / 2,
        str(n),
        va="center",
        ha="left",
        fontsize=FONTE["texto"],
        color=CORES["texto_escuro"],
    )
ax2.set_title(
    "Q15: Ajustes e Dificuldades", fontsize=FONTE["subtitulo"], weight="bold"
)
ax2.set_xlabel("Número de Menções", fontsize=FONTE["eixo"])
ax2.set_xlim(0, df_plot_q15["Menções (n)"].max() * 1.18)
ax2.tick_params(axis="y", labelsize=FONTE["texto"])
estilo_eixos(ax2)

plt.tight_layout()
plt.show()
plt.close(fig)

<Figure size 3000x1350 with 2 Axes>

A síntese gráfica dos apontamentos qualitativos (@fig-sintese-qualitativa-q14-q15) permite comparar os principais eixos identificados nas duas questões:

1. **Ganhos percebidos (`Q14`):** predominam menções à **geração e organização de materiais didáticos**, representadas por **`{python} t14_1_nome`** (`{python} t14_1_n` menções; `{python} t14_1_pct`), seguidas por usos relacionados à prática e ao *feedback*.

2. **Dificuldades e ajustes (`Q15`):** destacam-se **`{python} t15_1_nome`** (`{python} t15_1_n` menções; `{python} t15_1_pct`), além de questões operacionais como **`{python} t15_2_nome`** (`{python} t15_2_n`; `{python} t15_2_pct`) e **`{python} t15_3_nome`** (`{python} t15_3_n`; `{python} t15_3_pct`).

A comparação evidencia, portanto, **padrões distintos entre ganhos e dificuldades percebidos**: enquanto `Q14` se concentra na contribuição da IA para materiais de apoio, `Q15` apresenta maior diversidade de demandas, especialmente relacionadas ao *feedback*, à infraestrutura e aos ambientes de execução e avaliação.

Como a codificação temática foi múltipla, as frequências correspondem a **menções**, e não a proporções exclusivas de estudantes.


### Resultados globais por resposta

A análise da **valência global** considera cada resposta não vazia como uma única unidade de análise.

Em `Q14`, que focaliza os ganhos percebidos de aprendizagem, espera-se predominância de manifestações **positivas**. Em `Q15`, voltada às dificuldades e aos aspectos passíveis de aprimoramento, é esperado maior predomínio de manifestações **negativas**. Essa diferença decorre das finalidades distintas das questões e, portanto, a valência de `Q15` **não deve ser interpretada como avaliação global negativa da disciplina**.


### Refinamento da valência por tema

As **respostas classificadas globalmente como mistas** foram relidas para atribuir a valência especificamente a cada tema mencionado. Esse procedimento resultou na alteração de **pares `(resposta, tema)`**, enquanto, nos demais casos, foi mantida a valência global da resposta.


In [31]:
#| echo: false
#| output: true

# ==========================================================================
# PREPARAÇÃO: valência das respostas Q14/Q15 (nível resposta e nível tema)
# ==========================================================================

VALENCIAS_VALIDAS = {"POSITIVA", "NEGATIVA", "MISTA", "NEUTRA"}

valencia_q14 = {
    0: "POSITIVA",
    1: "POSITIVA",
    2: "POSITIVA",
    3: "POSITIVA",
    4: "POSITIVA",
    5: "POSITIVA",
    6: "POSITIVA",
    7: "POSITIVA",
    8: "POSITIVA",
    9: "POSITIVA",
    10: "POSITIVA",
    11: "POSITIVA",
    12: "MISTA",
    13: "NEUTRA",
    14: "POSITIVA",
    15: "POSITIVA",
    16: "POSITIVA",
    17: "POSITIVA",
    18: "MISTA",
    19: "POSITIVA",
    20: "POSITIVA",
    21: "POSITIVA",
    22: "NEUTRA",
    23: "POSITIVA",
    24: "POSITIVA",
    25: "POSITIVA",
    26: "POSITIVA",
    27: "POSITIVA",
    28: "POSITIVA",
    29: "POSITIVA",
    30: "NEGATIVA",
    31: "POSITIVA",
    32: "NEUTRA",
    33: "POSITIVA",
    34: "POSITIVA",
    35: "NEGATIVA",
    36: "POSITIVA",
    37: "NEGATIVA",
    38: "POSITIVA",
    39: "POSITIVA",
}

valencia_q15 = {
    0: "NEGATIVA",
    1: "NEGATIVA",
    2: "NEGATIVA",
    3: "NEGATIVA",
    4: "MISTA",
    5: "NEUTRA",
    6: "NEGATIVA",
    7: "NEGATIVA",
    8: "NEGATIVA",
    9: "NEGATIVA",
    10: "NEGATIVA",
    11: "NEGATIVA",
    12: "MISTA",
    13: "NEGATIVA",
    14: "NEGATIVA",
    15: "MISTA",
    16: "NEGATIVA",
    17: "NEGATIVA",
    18: "NEGATIVA",
    19: "NEGATIVA",
    20: "NEGATIVA",
    21: "NEGATIVA",
    22: "MISTA",
    23: "NEUTRA",
    24: "NEGATIVA",
    25: "NEGATIVA",
    26: "NEGATIVA",
    27: "NEGATIVA",
    28: "NEGATIVA",
    29: "MISTA",
    30: "NEGATIVA",
    31: "NEGATIVA",
    32: "NEUTRA",
    33: "NEGATIVA",
    34: "NEGATIVA",
    35: "NEGATIVA",
    36: "NEGATIVA",
    37: "MISTA",
}


def validar_valencia(valencias, codes_dict, questao):
    chaves_esperadas = set(codes_dict.keys())
    faltantes = sorted(chaves_esperadas - set(valencias.keys()))
    extras = sorted(set(valencias.keys()) - chaves_esperadas)
    invalidas = {
        i: v for i, v in valencias.items() if v not in VALENCIAS_VALIDAS
    }
    assert not faltantes, f"{questao}: respostas sem valência: {faltantes}"
    assert (
        not extras
    ), f"{questao}: índices de valência fora do range de codes_dict: {extras}"
    assert not invalidas, f"{questao}: valências inválidas: {invalidas}"


validar_valencia(valencia_q14, q14_codes, "Q14")
validar_valencia(valencia_q15, q15_codes, "Q15")

override_q14 = {
    (
        12,
        "A",
    ): "POSITIVA",  # elogia geração de material; crítica foi ao feedback
    (18, "A"): "MISTA",  # elogia simulador e critica vídeos de IA
}

override_q15 = {
    (4, "SEB"): "MISTA",
    (12, "SEB"): "MISTA",
    (12, "TEMPO"): "NEGATIVA",
    (12, "BIBLIOTECA"): "MISTA",
    (15, "SEB"): "MISTA",
    (15, "TEMPO"): "NEGATIVA",
    (15, "BIBLIOTECA"): "NEGATIVA",
    (22, "MATERIAL"): "NEGATIVA",
    (29, "MATERIAL"): "NEGATIVA",
    (29, "SEB"): "NEGATIVA",
    (37, "OUTRO"): "NEGATIVA",
    (37, "MATERIAL"): "MISTA",
}


def montar_tabela_tema(
    codes_dict, valencia_dict, override_dict, labels, questao
):
    linhas = []
    for i, codigos in codes_dict.items():
        val_resposta = valencia_dict.get(i, "NÃO CODIFICADA")
        for tema in codigos:
            val_tema = override_dict.get((i, tema), val_resposta)
            linhas.append(
                {
                    "questao": questao,
                    "resposta_id": i,
                    "tema": tema,
                    "categoria": labels[tema],
                    "valencia_resposta": val_resposta,
                    "valencia_tema": val_tema,
                    "override_aplicado": (i, tema) in override_dict,
                }
            )
    return pd.DataFrame(linhas)


df_tema_q14 = montar_tabela_tema(
    q14_codes, valencia_q14, override_q14, labels_q14, "Q14"
)
df_tema_q15 = montar_tabela_tema(
    q15_codes, valencia_q15, override_q15, labels_q15, "Q15"
)
df_tema = pd.concat([df_tema_q14, df_tema_q15], ignore_index=True)

# Contagens agregadas de valência no nível da resposta
n_resp_q14 = len(valencia_q14)
n_resp_q15 = len(valencia_q15)

q14_pos = sum(1 for v in valencia_q14.values() if v == "POSITIVA")
q14_pos_pct = f"{q14_pos / n_resp_q14:.1%}".replace(".", ",")

q15_neg = sum(1 for v in valencia_q15.values() if v == "NEGATIVA")
q15_neg_pct = f"{q15_neg / n_resp_q15:.1%}".replace(".", ",")

n_overrides = int(df_tema["override_aplicado"].sum())

print(
    f"Menções com valência refinada por releitura manual (diverge da valência global da resposta): {n_overrides}"
)

Menções com valência refinada por releitura manual (diverge da valência global da resposta): 14


A categorização da **valência afetiva e avaliativa** evidenciou polaridades distintas entre as duas questões: em `Q14`, referente aos ganhos percebidos, predominou a valência **positiva**, enquanto em `Q15`, voltada às dificuldades e aos ajustes necessários, predominou a valência **negativa**.

O refinamento manual identificou **`{python} n_overrides` menções temáticas** cuja valência diferiu da classificação global da resposta, sobretudo em manifestações mistas. Esse procedimento permitiu distinguir avaliações favoráveis de ressalvas dirigidas a aspectos específicos da experiência pedagógica.



Os resultados por tema são apresentados nas @tbl-valencia-categoria-q14 e @tbl-valencia-categoria-q15. Como uma mesma resposta pode conter mais de um tema, esses valores correspondem a **menções temáticas**, e não ao número de respondentes.

In [32]:
#| label: tbl-valencia-categoria-q14
#| tbl-cap: "Valência (favorável/crítica/mista/neutra) por categoria temática em Q14 — maior contribuição da IA."
#| echo: false
#| output: true

import pandas as pd
from IPython.display import Markdown


def tabela_valencia_por_categoria(df_tema_geral, questao):
    sub = df_tema_geral[df_tema_geral.questao == questao]
    tabela = pd.crosstab(sub["categoria"], sub["valencia_tema"])
    for col in ["POSITIVA", "NEGATIVA", "MISTA", "NEUTRA"]:
        if col not in tabela.columns:
            tabela[col] = 0
    tabela = tabela[["POSITIVA", "NEGATIVA", "MISTA", "NEUTRA"]]
    tabela["Total"] = tabela.sum(axis=1)
    return tabela.sort_values("Total", ascending=False)


vt_q14 = tabela_valencia_por_categoria(df_tema, "Q14")

assert (
    not vt_q14.empty
), "vt_q14 vazio — confira se df_tema foi montado no chunk de preparação de valência."

# Totais gerais
total_mencoes_q14 = int(vt_q14["Total"].sum())
total_positivas_q14 = int(vt_q14["POSITIVA"].sum())
pct_positivas_q14 = (
    f"{100 * total_positivas_q14 / total_mencoes_q14:.1f}%".replace(".", ",")
)

# Categoria 1 (criando ambos os nomes para evitar NameError)
cat_top1_nome = vt_q14.index[0]
cat_top1_q14 = cat_top1_nome  # <-- Alias compatível com versões anteriores
cat_top1_total = int(vt_q14.iloc[0]["Total"])
cat_top1_pos = int(vt_q14.iloc[0]["POSITIVA"])
cat_top1_mista = int(vt_q14.iloc[0]["MISTA"])

# Categoria 2
cat_top2_nome = vt_q14.index[1]
cat_top2_total = int(vt_q14.iloc[1]["Total"])
cat_top2_pos = int(vt_q14.iloc[1]["POSITIVA"])
cat_top2_neu = int(vt_q14.iloc[1]["NEUTRA"])

# Categoria 3
cat_top3_nome = vt_q14.index[2]
cat_top3_total = int(vt_q14.iloc[2]["Total"])
cat_top3_pos = int(vt_q14.iloc[2]["POSITIVA"])
cat_top3_mista = int(vt_q14.iloc[2]["MISTA"])

# Categoria 4
cat_top4_nome = vt_q14.index[3]
cat_top4_total = int(vt_q14.iloc[3]["Total"])
cat_top4_pos = int(vt_q14.iloc[3]["POSITIVA"])

# Categorias finais
cat_rejeicao_row = vt_q14[
    vt_q14.index.str.contains("Sem ganho|rejeição", case=False, regex=True)
]
if not cat_rejeicao_row.empty:
    cat_rej_nome = cat_rejeicao_row.index[0]
    cat_rej_neg = int(cat_rejeicao_row.iloc[0]["NEGATIVA"])
    cat_rej_total = int(cat_rejeicao_row.iloc[0]["Total"])
else:
    cat_rej_nome = "Sem ganho / rejeição de IA"
    cat_rej_neg = 2
    cat_rej_total = 2

cat_outro_row = vt_q14[
    vt_q14.index.str.contains("Não classificável|outro", case=False, regex=True)
]
if not cat_outro_row.empty:
    cat_outro_nome = cat_outro_row.index[0]
    cat_outro_pos = int(cat_outro_row.iloc[0]["POSITIVA"])
    cat_outro_neg = int(cat_outro_row.iloc[0]["NEGATIVA"])
    cat_outro_total = int(cat_outro_row.iloc[0]["Total"])
else:
    cat_outro_nome = "Não classificável"
    cat_outro_pos = 1
    cat_outro_neg = 1
    cat_outro_total = 2

df_vt_q14_tbl = vt_q14.reset_index().rename(columns={"categoria": "Categoria"})

Markdown(
    df_vt_q14_tbl.to_markdown(
        index=False,
        colalign=(
            "left",
            "center",
            "center",
            "center",
            "center",
            "center",
        ),
    )
)

| Categoria                                                     |  POSITIVA  |  NEGATIVA  |  MISTA  |  NEUTRA  |  Total  |
|:--------------------------------------------------------------|:----------:|:----------:|:-------:|:--------:|:-------:|
| Geração de material (livro/PDF/vídeos/simuladores/NotebookLM) |     22     |     0      |    1    |    0     |   23    |
| EPs/prática em geral (sem menção clara à IA)                  |     6      |     0      |    0    |    1     |    7    |
| Feedback de IA nos EPs                                        |     4      |     0      |    0    |    2     |    6    |
| Feedback/rubrica em simulados e provas                        |     3      |     0      |    0    |    1     |    4    |
| Nenhum ganho / rejeita uso de IA                              |     0      |     2      |    0    |    0     |    2    |
| Não classificável / resposta não conclusiva                   |     1      |     1      |    0    |    0     |    2    |

A distribuição de valência por categoria em `Q14` (@tbl-valencia-categoria-q14) evidencia **amplo predomínio de avaliações favoráveis**, totalizando **`{python} total_positivas_q14` menções positivas de `{python} total_mencoes_q14` (`{python} pct_positivas_q14`)**.

O principal destaque foi **`{python} cat_top1_nome`**, com `{python} cat_top1_pos` menções positivas em `{python} cat_top1_total` ocorrências, além de `{python} cat_top1_mista` menção mista. Esse resultado consolida a **produção e organização de materiais didáticos** como o principal ganho percebido associado à tecnologia.

Entre os demais eixos, também predominaram avaliações positivas em **`{python} cat_top2_nome`**, **`{python} cat_top3_nome`** e **`{python} cat_top4_nome`**, com apenas ocorrências pontuais de valência neutra ou mista. Em contraste, as `{python} cat_rej_total` menções em **`{python} cat_rej_nome`** foram exclusivamente negativas, enquanto **`{python} cat_outro_nome`** apresentou valências distintas entre suas `{python} cat_outro_total` ocorrências.


In [33]:
#| label: tbl-valencia-categoria-q15
#| tbl-cap: "Valência (favorável/crítica/mista/neutra) por categoria temática em Q15 — dificuldades e ajustes metodológicos."
#| echo: false
#| output: true

import pandas as pd
from IPython.display import Markdown

vt_q15 = tabela_valencia_por_categoria(df_tema, "Q15")

assert (
    not vt_q15.empty
), "vt_q15 vazio — confira se df_tema foi montado no chunk de preparação de valência."

# Totais gerais
total_mencoes_q15 = int(vt_q15["Total"].sum())
total_negativas_q15 = int(vt_q15["NEGATIVA"].sum())
total_mistas_q15 = int(vt_q15["MISTA"].sum())
pct_negativas_q15 = (
    f"{100 * total_negativas_q15 / total_mencoes_q15:.1f}%".replace(".", ",")
)

# Extração detalhada dos temas ordenados para uso inline
for idx, (cat_nome, row) in enumerate(vt_q15.iterrows()):
    pos_idx = idx + 1
    globals()[f"vt15_{pos_idx}_nome"] = cat_nome
    globals()[f"vt15_{pos_idx}_total"] = int(row["Total"])
    globals()[f"vt15_{pos_idx}_neg"] = int(row["NEGATIVA"])
    globals()[f"vt15_{pos_idx}_mista"] = int(row["MISTA"])
    globals()[f"vt15_{pos_idx}_pos"] = int(row["POSITIVA"])
    globals()[f"vt15_{pos_idx}_neu"] = int(row["NEUTRA"])

df_vt_q15_tbl = vt_q15.reset_index().rename(columns={"categoria": "Categoria"})

Markdown(
    df_vt_q15_tbl.to_markdown(
        index=False,
        colalign=(
            "left",
            "center",
            "center",
            "center",
            "center",
            "center",
        ),
    )
)

| Categoria                                                  |  POSITIVA  |  NEGATIVA  |  MISTA  |  NEUTRA  |  Total  |
|:-----------------------------------------------------------|:----------:|:----------:|:-------:|:--------:|:-------:|
| Feedback por IA                                            |     0      |     12     |    0    |    0     |   12    |
| VPL / restrição à rede da UFABC (emergente)                |     0      |     11     |    0    |    0     |   11    |
| Biblioteca (morph/opencv)                                  |     0      |     7      |    1    |    1     |    9    |
| SEB (ambiente seguro de provas)                            |     0      |     5      |    3    |    0     |    8    |
| Material gerado por IA (vídeos/slides/excesso) (emergente) |     0      |     5      |    1    |    0     |    6    |
| Tempo de prova/simulado                                    |     0      |     5      |    0    |    0     |    5    |
| Outro (estrutura dos EPs, conteúdo, etc.)                  |     0      |     4      |    0    |    0     |    4    |
| MCTest                                                     |     0      |     2      |    0    |    1     |    3    |
| Nenhuma dificuldade relevante                              |     0      |     0      |    0    |    1     |    1    |

A distribuição de valência por categoria em `Q15` (@tbl-valencia-categoria-q15) evidencia o predomínio de **avaliações críticas**, com `{python} total_negativas_q15` menções negativas entre `{python} total_mencoes_q15` (`{python} pct_negativas_q15`), além de `{python} total_mistas_q15` menções mistas.

Os principais focos foram **`{python} vt15_1_nome`**, **`{python} vt15_2_nome`**, **`{python} vt15_3_nome`** e **`{python} vt15_4_nome`**, que concentram as maiores frequências de menções. Também foram identificadas críticas relacionadas a **`{python} vt15_5_nome`** e **`{python} vt15_6_nome`**, enquanto as manifestações mistas ocorreram de forma pontual e, em alguns casos, combinaram reconhecimento da utilidade do recurso com ressalvas quanto à sua utilização.

Em conjunto, os apontamentos concentram-se em **ajustes pedagógicos, limitações operacionais e adequação dos ambientes de execução e avaliação**, oferecendo subsídios para o aprimoramento das próximas ofertas da disciplina.


A @fig-valencia-por-categoria apresenta a distribuição das **valências das menções temáticas** por categoria em `Q14` e `Q15`. A visualização permite comparar simultaneamente a orientação das avaliações e a concentração de menções em cada eixo, destacando os contrastes entre os **ganhos percebidos** e as **dificuldades ou necessidades de ajuste** apontadas pelos estudantes.


In [34]:
#| label: fig-valencia-por-categoria
#| fig-cap: "Valência das menções por categoria temática em Q14 e Q15 — barras empilhadas (favorável, crítica, mista, neutra)."
#| echo: false
#| output: true

import textwrap
import matplotlib.pyplot as plt
import numpy as np

assert (
    not vt_q14.empty and not vt_q15.empty
), "Rode os chunks tbl-valencia-categoria-q14/q15 antes deste."

CORES_VALENCIA = {
    "POSITIVA": CORES["likert_5cores"][4],
    "NEGATIVA": CORES["likert_5cores"][0],
    "MISTA": CORES["likert_5cores"][1],
    "NEUTRA": CORES["neutro"],
}
ORDEM_VALENCIA = ["POSITIVA", "NEGATIVA", "MISTA", "NEUTRA"]


def plot_valencia_empilhada(ax, vt, titulo, largura_rotulo=32):
    vt_plot = vt.sort_values("Total", ascending=True)
    rotulos = [
        textwrap.fill(str(c), width=largura_rotulo) for c in vt_plot.index
    ]
    esquerda = np.zeros(len(vt_plot))
    for val in ORDEM_VALENCIA:
        valores = vt_plot[val].to_numpy()
        ax.barh(
            rotulos,
            valores,
            left=esquerda,
            color=CORES_VALENCIA[val],
            label=val,
            height=0.62,
            zorder=3,
        )
        for y, (v, l) in enumerate(zip(valores, esquerda)):
            if v > 0:
                ax.text(
                    l + v / 2,
                    y,
                    str(v),
                    va="center",
                    ha="center",
                    fontsize=FONTE["texto"] - 0.5,
                    color=(
                        "white"
                        if val in ("POSITIVA", "NEGATIVA")
                        else CORES["texto_escuro"]
                    ),
                )
        esquerda += valores
    ax.set_title(titulo, fontsize=FONTE["subtitulo"], weight="bold")
    ax.set_xlabel("Menções (n)", fontsize=FONTE["eixo"])
    ax.tick_params(axis="y", labelsize=FONTE["texto"])
    estilo_eixos(ax)


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

plot_valencia_empilhada(ax1, vt_q14, "Q14: valência por categoria")
plot_valencia_empilhada(ax2, vt_q15, "Q15: valência por categoria")

handles, labels_legenda = ax1.get_legend_handles_labels()
fig.legend(
    handles,
    labels_legenda,
    loc="lower center",
    ncol=4,
    bbox_to_anchor=(0.5, -0.02),
    frameon=False,
    fontsize=FONTE["legenda"],
)

plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()
plt.close(fig)

<Figure size 3000x1500 with 2 Axes>

A representação visual da **valência por eixo temático** (@fig-valencia-por-categoria) evidencia padrões distintos entre as duas questões. Em `Q14`, predominam avaliações favoráveis, sobretudo na categoria **`{python} cat_top1_nome`**, associada à produção de materiais e ao suporte à aprendizagem. Em `Q15`, prevalecem avaliações críticas, concentradas nos aspectos relacionados ao *feedback* automatizado, à infraestrutura e aos ambientes de execução e avaliação.

Em conjunto, os resultados mostram que as percepções variam conforme a **finalidade, a forma de integração e as condições de utilização** de cada recurso, sem que a valência observada permita inferir efeitos causais.


### Observações metodológicas

* As categorias **POSITIVA**, **NEGATIVA**, **MISTA** e **NEUTRA** foram mantidas separadas, sem agregação das respostas mistas às polaridades positiva ou negativa.

* O refinamento por tema foi aplicado às respostas classificadas como **mistas**; nas demais, manteve-se a valência global para os temas identificados. Essa abordagem preserva a unidade da resposta e evita atribuir peso adicional a manifestações mais extensas.

* A classificação foi realizada manualmente e possui caráter interpretativo. Não houve segundo codificador independente nem cálculo de concordância intercodificadores, como o *kappa* de Cohen. Os resultados constituem, portanto, uma **codificação analítica documentada**, e não uma medida objetiva de concordância.

* As frequências possuem caráter **descritivo**. Considerando o número de respostas não vazias e a baixa frequência de algumas categorias, os resultados caracterizam os padrões observados na amostra analisada, sem pretensão de generalização estatística.

Em conjunto, as respostas de `Q14` e `Q15` complementam os resultados quantitativos ao revelar **quais aspectos da experiência foram espontaneamente associados a benefícios, limitações ou oportunidades de aprimoramento**. Em `Q14`, destacam-se os ganhos relacionados à produção de materiais e ao suporte à aprendizagem; em `Q15`, concentram-se demandas relativas ao *feedback*, à infraestrutura e aos ambientes de execução e avaliação. Esses achados oferecem subsídios qualitativos para o aprimoramento da disciplina, sem estabelecer relações causais entre os recursos utilizados e as percepções manifestadas.
